To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Introducing **[Unsloth Desktop](https://unsloth.ai/docs/desktop)**, the first desktop app to run and train models. Free and open-source for macOS, Windows and Linux. [GitHub](https://github.com/unslothai/unsloth) • [Download](https://unsloth.ai/download)

<p>
<a href="https://unsloth.ai/docs/desktop"><img src="https://raw.githubusercontent.com/unslothai/notebooks/refs/heads/main/assets/unsloth-qwen3-8.png" width="350" alt="Introducing Unsloth Desktop"></a>
</p>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [1]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

### Unsloth

In [19]:
from unsloth import FastLanguageModel # FastLanguageModel for LLMs
import torch

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Llama-3.2-11B-Vision-Instruct-bnb-4bit", # Llama 3.2 vision support
    "unsloth/Llama-3.2-11B-Vision-bnb-4bit",
    "unsloth/Llama-3.2-90B-Vision-Instruct-bnb-4bit", # Can fit in a 80GB card!
    "unsloth/Llama-3.2-90B-Vision-bnb-4bit",

    "unsloth/Pixtral-12B-2409-bnb-4bit",              # Pixtral fits in 16GB!
    "unsloth/Pixtral-12B-Base-2409-bnb-4bit",         # Pixtral base model

    "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit",          # Qwen2 VL support
    "unsloth/Qwen2-VL-7B-Instruct-bnb-4bit",
    "unsloth/Qwen2-VL-72B-Instruct-bnb-4bit",

    "unsloth/llava-v1.6-mistral-7b-hf-bnb-4bit",      # Any Llava variant works!
    "unsloth/llava-1.5-7b-hf-bnb-4bit",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    "Qwen/Qwen3.5-0.8B",
    load_in_4bit = False, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

==((====))==  Unsloth 2026.10.10: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters.

**[NEW]** We also support finetuning ONLY the vision part of the model, or ONLY the language part. Or you can select both! You can also select to finetune the attention or the MLP layers!

In [20]:
#model = FastVisionModel.get_peft_model(
#    model,
#    finetune_vision_layers     = False, # False if not finetuning vision layers
#    finetune_language_layers   = True, # False if not finetuning language layers
#    finetune_attention_modules = True, # False if not finetuning attention layers
#    finetune_mlp_modules       = True, # False if not finetuning MLP layers
#
#    r = 16,           # The larger, the higher the accuracy, but might overfit
#    lora_alpha = 16,  # Recommended alpha == r at least
#    lora_dropout = 0,
#    bias = "none",
#    random_state = 3407,
#    use_rslora = False,  # We support rank stabilized LoRA
#    loftq_config = None, # And LoftQ
#    # target_modules = "all-linear", # Optional now! Can specify a list if needed
#)

model = FastLanguageModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


In [21]:
test_data = [
  {
    "instruction": "\nYou're an expert text parser. Your goal is to parse TSV fields from the input text. A single input text file can have multiple \ndifferent species and/or multiple individuals from the same species. The output should include a TSV header and the following\nfields should be filled for each species with information found from the input text:\n\nreferences:\nDescription: Complete bibliographic reference formatted according to Harvard citation style.\nExamples: Smith, J., Doe, A. and Jones, B. (2020) 'Plant functional traits across biomes', Journal of Ecology, 108(4), pp. 1200-1215.\n\nverbatimScientificName:\nDescription: The full, original scientific name as supplied in the raw data, including authorship or subspecific rank if recorded.\nExamples: Pinus sylvestris L., Panthera leo (Linnaeus, 1758)\n\ntaxonRank:\nDescription: Taxonomic rank of the target organism.\nExamples: species, genus, subspecies\n\nverbatimTraitName:\nDescription: Exact trait name as originally recorded in the source data or publication.\nExamples: SLA, specific leaf area, body mass\n\nverbatimTraitUnit:\nDescription: Original unit of measurement as stated in the source.\nExamples: mm, g, mm2/mg, kg\n\nindividualCount:\nDescription: Number of individuals aggregated into the reported measurement.\nExamples: 1, 15, 100\n\nmeasurementValue_min:\nDescription: Minimum measured value within a range or aggregate group.\nExamples: 12.4, 0.5\n\nmeasurementValue_max:\nDescription: Maximum measured value within a range or aggregate group.\nExamples: 28.1, 3.2\n\ndispersion:\nDescription: Measure of variation/dispersion associated with the central value (e.g., standard deviation, standard error).\nExamples: SD = 1.2, 0.45\n\nstatisticalMethod:\nDescription: Statistical metric used to summarize the values.\nExamples: mean, median, mode, range\n\nverbatimTraitValue:\nDescription: Raw value or text string as originally reported (numerical or categorical).\nExamples: 15.4, herbaceous, red\n\nsex:\nDescription: Biological sex of the specimen(s).\nExamples: male, female, hermaphrodite, mixed\n\nlifeStage:\nDescription: Developmental age or life stage of the organism.\nExamples: adult, juvenile, seedling, larva\n\nmeasurementMethod:\nDescription: Protocols, instruments, or procedures used to capture the measurement.\nExamples: Swinbank dry weight method, Caliper measurement of tarsus length\n\nmeasurementRemarks:\nDescription: Free-text notes or qualifying details about the measurement process or state of the specimen.\nExamples: Measured on dried museum specimens, Leaves collected from upper canopy\n\nmeasurementAccuracy:\nDescription: Precision, error tolerance, or accuracy rating of the instrument/method used.\nExamples: ± 0.01 g, 0.1 mm\n\nmeasurementDeterminedBy:\nDescription: Name or ORCID of the person/entity who made or validated the measurement.\nExamples: Jane Doe, https://orcid.org/0000-0000-0000-0000\n\nverbatimLocality:\nDescription: Original description of the sampling location before standardized georeferencing.\nExamples: Mount Rainier National Park, trail to Paradise, Plot 4B, Black Forest\n\nauthor:\nDescription: This field has to only include bioparser\nExamples: BioParser, BioParser, BioParser\n\nassociatedReferences:\nDescription: Citations, DOIs, or bibliographic references supporting the dataset or measurement.\nExamples: https://doi.org/10.1111/1365-2435.12345, Smith et al. 2020\n\nFULL OUTPUT EXAMPLE:\n[['references', 'verbatimScientificName', 'taxonRank', 'verbatimTraitName', 'verbatimTraitUnit', 'individualCount', 'measurementValue_min', 'measurementValue_max', 'dispersion', 'statisticalMethod', 'verbatimTraitValue', 'sex', 'lifeStage', 'measurementMethod', 'measurementRemarks', 'measurementAccuracy', 'measurementDeterminedBy', 'verbatimLocality', 'author', 'associatedReferences'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'BM = Body Mass in grams (g)', 'g ', '', '4.9', '5.8', '', '', '5.2', 'male', '', '', '', '', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'BM = Body Mass in grams (g)', 'g ', '', '4.5', '5.4', '', '', '4.8', 'female', '', '', '', '', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'TBL = Total Body Length', 'mm ', '', '152', '175', '', '', '166.3', 'male', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'WS = Wing Span (width of the bat)', 'mm ', '', '152', '175', '', '', '166.3', 'male', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'WS = Wing Span (width of the bat)', 'mm ', '', '118', '168', '', '', '149.5', 'female', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'FA = Forearm length', 'mm', '', '32', '34.6', '', '', '33.2', 'male', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'FA = Forearm length', 'mm', '', '32.4', '33.3', '', '', '32.9', 'female', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'D5 = 5th Digit length', 'mm', '', '40', '44.5', '', '', '41.9', 'male', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'D5 = 5th Digit length', 'mm ', '', '40', '41.4', '', '', '40.5', 'female', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'D3 = 3rd Digit length', 'mm ', '', '54', '58.6', '', '', '56.2', 'male', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'D3 = 3rd Digit length', 'mm ', '', '54.1', '57.2', '', '', '55.1', 'female', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'TL = Tail Length', 'mm ', '', '27.5', '36.4', '', '', '30.7', 'male', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'TL = Tail Length', 'mm ', '', '21.4', '31.5', '', '', '27.8', 'female', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'TBL = Total Body Length', 'mm ', '63', '60.3', '76.4', '', '', '', '', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'TL = Tail Length', 'mm ', '63', '21.4', '36.4', '', '', '', '', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'WS = Wing Span (width of the bat)', 'mm ', '63', '118', '175', '', '', '', '', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'FA = Forearm length', 'mm ', '63', '32', '34.6', '', '', '', '', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'D5 = 5th Digit length', 'mm ', '63', '40', '44.5', '', '', '', '', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'D3 = 3rd Digit length', 'mm ', '63', '54', '61', '', '', '', '', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'BM = Body Mass in grams (g)', 'g ', '63', '4.5', '5.8', '', '', '', '', '', '', '', '', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'EL = Ear Length', 'mm ', '63', '8.2', '9.6', '', '', '', '', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study']]\"\n",
    "input": "Original article\n\n# Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats\n\n![](images/page_0_image_9.jpg)\n\nFouad Abdel-Hamid a,⇑, Abdulaziz R. Alqahtani b\n\n<sup>a</sup> Departments of Zoology, Faculty of Science, Al-Azhar University in Cairo, Egypt <sup>b</sup> Department of Biology, College of Science, University of Bisha, Bisha, Saudi Arabia\n\n## a r t i c l e i n f o\n\nArticle history: Received 20 December 2021 Revised 22 September 2022 Accepted 25 October 2022 Available online 1 November 2022\n\nKeywords: Morphological Feeding habits Insectivorous Pipistrelle Roosting and foraging\n\n## a b s t r a c t\n\nThis study revises the morphological characters of both sexes of Kuhl’s pipistrelle, including external, cranial and dental characters. This study also establishes a baseline for the diet of this insectivorous bat species across a variety of habitats in the Nile Valley, where diets have never before been investigated. Our findings of the stomach contentswere not significantly affected despite the different habitats from which the samples were collected. The three most common insect taxa recovered were Hymenoptera, Lepidoptera and Coleoptera. The percentage volume of insect orders in the diet of Kuhl’s bat (63 stomachs) consisted of 29% of Hymenoptera, 9% of Coleoptera, 27% of Lepidoptera, 32% unknown materials and unexpected thing of these insectivorous bats was the presences of 3% fruits. A single order did not dominate the diet of this specie because Hymenoptera was the most fed insect orders by frequency of occurrence 87.3%, Lepidoptera by 74.6%, Coleoptera 36.5% and fruits in 4.8% of samples. Also, they may also be fed by other insect orders, as unknown materials were found in all samples. \u0001 2022 The Author(s). Published by Elsevier B.V. on behalf of King Saud University. This is an open access © 2022 The Author(s). Published by Elsevier B.V. on behalf of King Saud University. This is an open access article under the CC BY-NC-ND license ([http://creativecommons.org/licenses/by-nc-nd/4.0/](http://creativecommons.org/licenses/by-nc-nd/4.0/)).\n\n## 1. Introduction\n\nBats (Chiroptera) are one of the most diverse and species-rich terrestrial vertebrate orders on the planet because they live in almost all environments, from rainforests to deserts, and all in between, ranging from subarctic to tropical latitudes as well as they use a variety of foraging strategies, social habits, vocal communication, navigational tools and learning abilities (Amichai, 2017).\n\nEgypt’s mammalian fauna has been studied since the Linnaean era, and many species of mammals, including bats, have now been studied and described. Bats have been the subject of several studies over the last two centuries, according to (Benda et al. 2008), and expertise in the fields of taxonomy, ecology, and species distribution has evolved over time.. Osborn and Helmy (1980) describe Egypt’s mammalian fauna, which included Egyptian bats from all over Egypt, including Nubia and Sinai, following that, many research on bats. The most reliable summary of Egyptian bat records was Qumsiyeh’s (1985) review; then Harrison and Bates (1991) book ‘‘Mammals of Arabia” which included bat taxonomy, ecology, and species distribution in Arabia. Wassif, 1995 paper describe many Egyptian bats from all over Egypt, including Sinai, and the identification key of Egyptian bats by (Dietz, 2005), while the most recent paper on bats was published by (Benda et al. 2008).\n\nUntil now, bat research in this region has lacked information regarding the biology of the bat species occuring in Egypt, and there is no available information on diet. Here, we focus on Kuhl’s pipistrelle, Pipistrellus kuhlii, it’s a member of family Vespertilionidae and it’s one of the most common and abundant bat species distributed in the Mediterranean basin of the Palaearctic region, and Middel East (Bray et al. 2013; Dalhoumi et al. 2018). Its range covers most of the Arabian Peninsula and extends to southwestern Asia in East (Sharifi et al. 2004). The distribution of the Kuhl’s pipistrelle bat is expanding in Northern Africa and in the Middle East (Bar-David et al. 2015). It occupies mainly lowlands, coastal areas, river valleys, but also urban areas such as towns and suburbs (Qumsiyeh, 1985; Mendelssohn and Yom-Tov 1999; Benda et al. 2006; WaWrocka et al., 2012). Due to this expansion in distribution range, P. kuhlii can be recognized as an adaptable generalist that easily adapted in urbanized environments (Russo and Ancillotto 2015; Ancillotto et al. 2015), this expansion show that Kuhl’s pipistrelle bat is probably behaviorally and physiologically adept to adjusting climate conditions outside its optimal range (Amichai and Korine 2020).\n\n<small><span class=\"docvortex-page-footnote\" data-block-type=\"page_footnote\" style=\"color:#6b7280\">⇑ Corresponding author.</span></small>\n\n<small><span class=\"docvortex-page-footnote\" data-block-type=\"page_footnote\" style=\"color:#6b7280\">Production and hosting by Elsevier</span></small>\n\n<small><span class=\"docvortex-page-footnote\" data-block-type=\"page_footnote\" style=\"color:#6b7280\">Peer review under responsibility of King Saud University.</span></small>\n\nThe knowledge of an animal’s diet is critical for understanding its ecological function as prey or as a predator, and its effect on local ecosystems. Since natural habitats are being altered as a result of increased urbanization and modern agricultural practices, this understanding is particularly important (Whitker et al. 1994). According to WaWrocka et al. (2012) the ability of animals to successfully forage is affected by the abundance and availability of their prey. Prey abundance (frequency and distribution in the environment), for insectivorous Kuhl’s bat has been defined as a s selective opportunist whereas it doesn’t show prefer a specific insects or feeding habitat, it tends to feed on insect-rich patches and to prefer larger prey items over smaller ones according to prey availability (Goiti et al. 2003). So, the diet composition of this bat can differ between seasons and habitats for a variety of reasons, including weather, prey availability, and population cycles.\n\nThis small bat species considered as a generalist insectivore, and its high behavioral flexibility, in much of its range, may utilize variety of cluttered environments (Kalko and Schnitzler 1993). It has been detected hunting over open natural land and agricultural fields, above water ponds, at vegetation edge and attracted to insects on street lights in rural and urban areas (Kalko and Schnitzler 1993; Ancillotto et al. 2015, 2016; Kahnonitch et al. 2018).\n\n## 2. Materials and methods\n\n## 2.1. Sampling\n\nPipistrelle bat Samples were collected from 10 roosting and foraging sites (n = 63, nearly 6 per location) collected from different habitats, which represented agricultural lands, fields, and densely populated urban areas in the governorates of Beni Suef and Minya Fig. 1. These habitats were visited in the Middle Nile Valley in\n\nEgypt, an area characterized by the narrow flood plain (fertile areas cultivated on both sides of the Nile River) and bounded on the eastern and western sides by vast desert. Mist nets were used to catch bats, according to the Indiana Bat Protocol (USFWS, 1999). Net sets were installed over water sources or near edge habitats, as well as in interior agriculture fields, according to (Carroll et al. 2002; Feldhamer et al. 2001). For each individual bat captured, its species, gender and body mass were recorded.\n\n## 2.2. Morphological analysis\n\nAll morphological characters used in this study is a descriptive data for species identification and its including: Eight external measurements were taken for each individual bat using a 0.1 mm caliper, according to (Benda et al. 2008). These were; FA = Forearm length, D5 = 5th Digit length, D3 = 3rd Digit length, TBL = Total Body Length, TL = Tail Length, WS = Wing Span (width of the bat), EL = Ear Length and HF = Hind Foot length. The ninth external character measured was BM = Body Mass in grams (g). And eight cranial measurements were taken for each skull using a 0.1 mm caliper, according to (Benda et al. 2008). The measurements were; GLS = Greatest Length of Skull, CBL = Condylobasal Length, CCL = Condylocanine Length, ZB = Zygomatic Breadth, SL = Snout Length, BL = Bulla Length, BW = Bulla Width and RWC = Width of Canine Row. An additional twelve dental measurements were taken for each bat’s jaw by using a 0.1 mm caliper. $\\mathbf { \\mathbf { \\mathbf { \\mathbf { \\mathbf { \\mathbf { \\mathbf { \\mathbf { \\mathbf { \\mathbf { \\mathbf { \\mathbf { \\mathbf { \\mathbf { \\mathbf { } } { M } } } } } } } } } } { \\mathbf { \\mathbf { \\mathbf { \\mathbf { \\mathbf { \\mathbf { \\mathbf { \\mathbf { \\mathbf { \\mathbf { \\mathbf { \\mathbf { \\mathbf { \\mathbf { \\mathbf { \\mathbf { } } } { } } } } } } } } } } } } } { \\mathbf { \\mathbf { \\mathbf { \\mathbf { \\mathbf { \\mathbf { \\mathbf { \\mathbf { \\mathbf { \\mathbf { \\mathbf { \\mathbf { \\mathbf { \\mathbf { } } } } } } } } } } } } } } } } ^ 3 } } } }$ = length of upper tooth-row between I1 and M3, $\\mathbf { C M } ^ { 2 } =$ length of upper tooth-row between C and M2, $\\mathbf { C M } ^ { 3 } =$ length of upper tooth-row between C and $\\mathbf{M}^{3}, \\mathbf{M}^{1} \\mathbf{M}^{3} =  length$ of upper molarsrow between $\\mathbf { M } ^ { 1 }$ and $\\mathbf { M } ^ { 3 } ,$ CPm1 = length of upper tooth-row between C and pm<sup>1</sup>, CML = Condylar Mandible Length, CPH = Coronoid Process Height, IM<sub>3</sub> = length of lower tooth-row between $\\mathrm { I } _ { 1 }$ and $\\mathbf { M } _ { 3 } ,$ CM3 = length of lower tooth-row between C and M3, $\\mathbf{M_{1}M_{3}} =$ length of lower molars-row between $\\mathbf { M } _ { 1 }$ and $\\mathbf { M } _ { 3 } ,$ $\\mathrm{CPm_{2}=length}$ of lower tooth-row between C and $\\mathtt { p m } _ { 2 }$ and $\\mathbf { C P m _ { 1 } } = -$ length of lower tooth-row between C and pm<sub>1</sub>.\n\n![](images/page_1_image_11.jpg)\n\nFig. 1. A map showing Study Area and Collection Localities of Kuhl’s pipistrelle bat samples.\n\n## 2.3. Stomach contents analysis\n\nWhole stomachs and their contents were removed from bats while they were freshly dead. Stomach contents were weighed to the nearest 0.1 mg and preserved in 70 % ethanol before analysis. Contents were analyzed in the laboratory to determine their food components and the percent volume of different dietary items.\n\n![](images/page_2_image_4.jpg)\n\nFig. 2. The preserved skin of Kuhl’s pipistrelle bat.\n\n![](images/page_2_image_6.jpg)\n\nUndigested prey remains (e.g., invertebrate exoskeletons, heads, antennae, mouthparts, prothoraxes, pterothoraxes, wings, legs, and other fragments) were compared to a reference collection to identify food items. All parts were examined under a stereoscopic microscope. By comparing parts to whole specimens, all insect remains were identified to order level using a variety of keys (Whitker, 1988; 2004; McAney et al. 1991; Borror et al. 1989). The value of each prey species in the diet was determined by its frequency (F) of occurrence (percentage of samples in which a given prey species was found) (Karanth and Sunquist 1995; Nunez et al. 2000; Taber et al. 1997).\n\n## 3. Results\n\n## 3.1. Descriptive morphological characters\n\nExternal morphology is widely used as descriptive data to distinguish bat species as well as to determine differences between individuals of the same species of both sexes. Based on the external characters taken for each individual of both sexes of Kuhl’s pipistrelle bats (Fig. 2), total body length ranged between 60.3 and 76.4 mm, and tail length: 21.4–36.4 mm, width of the bat (wings length): 118–175 mm, the forearm length: 32–34.6 mm, 5th digit: 40–44.5 mm, and 3rd digit: 54–61 mm. The weight of Kuhl’s pipistrelle bats ranged from 4.5 to 5.8 gm. Pelage dense, fine and short, fur color were very variable but were usually dark, ventral hairs with white tips and black bases whereas the dorsal hairs were buff tipped and had deep black bases, and naked parts of skin dark brown to blackish. Body color was variable with habitats, and their ears lengths ranged between 8.2 and 9.6 mm. Upper edge was narrow, front margin was nearly erect and the upper tooth row length CM3 was more than 4.5 mm.\n\n![](images/page_2_image_12.jpg)\n\nFig. 3. The skull of Kuhl’s pipistrelle bat, A) ventral view, B) lateral view, C) occipital view and D) lower jaw.\n\nBased on the external characters of Kuhl’s pipistrelle bats, the male individuals of this species are slightly larger than females in most external features (Annex 1). The mean weight of males was 5.2 g and ranged between 4.9 and 5.8 g while the mean weight of females was 4.8 g and ranged between 4.5 and 5.4 g. The mean body length of males (TBL) was 166.3 mm and ranged between 152 and 175 mm while the mean width of females was 149.5 mm and ranged between 118 and 168 mm. The mean width of males (WS) was 166.3 mm and ranged between 152 and 175 mm while the mean width of females was 149.5 mm and ranged between 118 and 168 mm. The mean forearm length of males was 33.2 mm, ranging between 32 and 34.6 mm while the mean forearm length of females was 32.9 mm, ranging between 32.4 and 33.3 mm. The mean length of the 5th digit of males was 41.9 mm, ranging between 40 and 44.5 mm while the mean length of the 5th digit of females was 40.5 mm and ranged between 40 and 41.4 mm. The mean length of the 3rd digit of males was56.2 mm and ranged between 54 and 58.6 mm while the mean length of the 3rd digit of females was 55.1 mm and ranged between 54.1 and 57.2 mm. The mean tail length of males was 30.7 mm and ranged between 27.5 and 36.4 mm while the mean tail length of females was 27.8 mm and ranged between 21.4 and 31.5 mm. One exception from the above results of external characters was the finding that ear length and hind foot length were slightly larger in females than in males (Annex 1) (see Fig. 3).\n\nNumber of occurrence and percentage of Frequency (F) and Volume (V) of food items in the stomach contents of Kuhl’s pipistrelle bat samples from each habitats.\n\n<table><tr><td>Food Items</td><td colspan=\"3\">Agricultural lands</td><td colspan=\"3\">Urban areas</td></tr><tr><td></td><td>No</td><td>Frequency (F)</td><td>Volume (V)</td><td>No</td><td>Frequency (F)</td><td>Volume (V)</td></tr><tr><td>Hymenoptera</td><td>30</td><td>88 %</td><td>30 %</td><td>25</td><td>86.2 %</td><td>29 %</td></tr><tr><td>Coleoptera</td><td>12</td><td>35.2 %</td><td>10 %</td><td>11</td><td>38 %</td><td>8%</td></tr><tr><td>Lepidoptera</td><td>25</td><td>73.5 %</td><td>27 %</td><td></td><td>75.9 %</td><td>25 %</td></tr><tr><td>Fruits</td><td>-</td><td>0%</td><td>0%</td><td>2329</td><td>10.3 %</td><td>8%</td></tr><tr><td>Unknown materials</td><td>34</td><td>100 %</td><td>33 %</td><td></td><td>100 %</td><td>30 %</td></tr></table>\n\n![](images/page_3_chart_7.jpg)\n\nFig. 4. Percentage of the Frequency occurrence (F) of food items in the stomach contents of Kuhl’s pipistrelle bat samples from each habitats.\n\n![](images/page_3_chart_9.jpg)\n\nFig. 5. Percentage of the Volume (V) of food items in the stomach contents of Kuhl’s pipistrelle bat samples from each habitats.\n\nNevertheless, the cranial and dental characters were taken for individuals of both sexes of Kuhl’s pipistrelle bats showed that, the males and females of this species were shown to be nearly equal in most cranial and dental features (Fig. 3).\n\n## 3.2. Food and feeding habits\n\nOur microscopic examination of the stomach content samples from each habitats, showed that, the general composition of the bat’s diet in agricultural lands were the Hymenoptera which showed the highest frequency (F) of occurrence of all insect orders in the diet, found in 30 stomachs equal 88 % of 34 bats samples were collected from agricultural lands, the volume of insects of this order in the diet of Kuhl’s bat was 30 %, while the prey species belonging to Lepidoptera were the second most frequently recognized insect order, found in 25 stomachs and 73.5 % of all bat samples, which represented 27 % of the total volume. Coleoptera were the third detected prey of all the insect orders, present in the stomachs of 12 bats and 35.2 % of all bats samples, representing 10 % of the volume of the bat’s diet in agricultural lands. Kuhl’s bat species may also feed on other insect orders, not identified here, because the presence of unknown digested materials in all samples which equal 33 % of the volume of stomach contents (Table 1 and Figs. 4 and 5).\n\nNumber of occurrence and percentage of Frequency (F) and Volume (V) of food items in the stomach contents of all Kuhl’s pipistrelle bat samples.\n\n| Food Items | no | Frequency (F) | Volume (V) |\n| --- | --- | --- | --- |\n| Hymenoptera | 55 | 87 % | 29 % |\n| Coleoptera | 23 | 36.50 % | 9 % |\n| Lepidoptera | 47 | 74.60 % | 27 % |\n| Fruits | 3 | 4.80 % | 3 % |\n| Unknown materials | 63 | 100 % | 32 % |\n\nOn the other hand, the general composition of the bat’s diet in the urban areas were the Hymenoptera which showed the highest frequency (F) of occurrence of all insect orders in the diet, found in 25 stomachs equal 86.2 % of 29 bats samples were collected from the urban areas, the volume of insects of this order in the diet of Kuhl’s bat was 29 %, while the prey species belonging to Lepidoptera were the second most frequently recognized insect order, found in 22 stomachs and 75.9 % of all bat samples in this habitats, which represented 25 % of the total volume. Coleoptera were the third detected prey of all the insect orders, present in the stomachs of 11 bats and 38 % of all bats samples, representing 8 % of the volume of the bat’s diet in the urban areas. One of the most surprising result was the presence of fruits remains in three samples, equaling 10.3 % of the bats sampled in the urban areas and representing 8 % of the volume of stomach contents on average. Also, we cannot be certain that this species does not feed on other insect orders, not identified here, because the presence of unknown digested materials in all samples which equal 30 % of the volume of stomach contents (Table 1 and Figs. 4 and 5).\n\nThrough the microscopic examination of the stomach content samples in above results, we found that, despite the different habitats from which the samples were collected, they did not significantly affect the type of insects that were identified in the stomach contents at the level of insect order. There may have been differences at the level of the insect species within the same order. However, the species-level identification of insects from stomach contents is very difficult because specimens are often partially digested. As a result, the dietary findings at the order level were nearly consistent across different habitats.\n\nThe results of the stomach content samples from both habitats in (Table 2; Figs. 6 and 7), showed that, the general composition of the bat’s diet, expressed as the number of stomachs containing the food item, its frequency (F) of occurrence (its percentage of all samples) and the volume of all food items found.\n\nAccording to our finding Hymenoptera showed the highest frequency (F) of occurrence of all insect orders in the diet, found in 55 stomachs and 87.3 % of all bats sampled under this study, the percentage volume of insects of this order in the diet of Kuhl’s bat was 29 %. While the prey species belonging to Lepidoptera were the\n\n![](images/page_4_chart_13.jpg)\n\nFig. 6. Percentage of the Frequency occurrence (F) and Volume (V) of food items in the stomach contents of Kuhl’s pipistrelle bat samples.\n\n![](images/page_5_image_2.jpg)\n\nFig. 7. Some insect’s parts and another food items under microscope a,b) represented Hymenopteran parts, c,d) Coleopteran parts, e,f) Lepidoptera insects parts and g,h) represented the fruit remains in the stomach contents of Kuhl’s pipistrelle bat samples.\n\nsecond most frequently detected insect order, found in 47 stomachs and 74.6 % of all bat samples. The percentage volume of Lepidopteran insects in the diet of Kuhl’s bat was 27 %. Coleoptera were the third most frequently detected prey of all the insect orders, present in the stomachs of 23 bats and 36.5 % of all bats samples, representing 9 % of the volume of insects in the diet. Kuhl’s bat species may also feed on other insect orders, not identified here, because the presence of unknown digested materials in all samples which equal 32 % of the volume of stomach contents (Table 2; Figs. 6 and 7).\n\nThese results reveal that the diet composition of the Kuhl’s bat species varies widely, and not dominated by a single order of insects, but rather than this species prefers the species of some insect orders more than others. Also this may be due to the abundance of different prey orders and their availability as prey during the main foraging activity times of this small bat. One of the most surprising result was the presence of fruits in three samples, equaling 4.8 % of the bats sampled in our study and representing 3 % of the volume of stomach contents on average. Consuming fruit may be a result of an adaptation by these insectivorous bats living in urban areas. It may be some bats favored foraging on insects above fruit remains in the trash, or alternatively, bats may be collecting insects from nearby fruit gardens.\n\n## 4. Discussion\n\nAccording to our descriptive morphological findings, we confirmed that the samples collected in the field belong to Pipistrellus kuhlii, as all measurements were compatible and within the limits of previous studies to identify this species, such as Qumsiyeh (1985) and the most recent identification key of Egyptian bats by (Dietz, 2005) and compatible with many measurements were taken from Europe and Eastern Mediterranean (Mendelssohn and Yom-Tov 1999; Dietz and von Helversen 2004; Benda et al. 2006). Whereas the Pipistrellus kuhlii upper tooth row length (CM3) must be greater than 4.5 mm according to Egyptian key, our results range from 4.5 mm to 4.9 mm, according to Egyptian key and Europe and Eastern Mediterranean key respectively, forearm length (FA) must be 30.7 – 37.4 mm and 31–40 mm, our results range from 32 mm to 34.6 mm, length of 5th digit (D5) was 40 – 45 mm and 40–45 mm, our results range from 40 mm to 44.5 mm, and length of 3rd digit (D3) was 54 – 61 mm and 54–61 mm, our results range between 54 mm and 58.6 mm. On the other hand, some measurements show a slightly difference with many measurements were taken after (Mendelssohn and Yom-Tov 1999; Dietz and von Helversen 2004; Benda et al. 2006) as in Greater length of skull range between 12.3 and 13.9 mm, our results range between 12.4 mm and 12.9 mm and Zygomatic breadth range between 7.8 and 9.3 mm, our results range between 7.0 mm and 7.4 mm. Also, by comparing the measurements of external and morphological characteristics, it was revealed that the male individuals of Kuhl’s pipistrelle bats are slightly larger than females with regards to most external features but are nearly identical in most cranial and dental characters.\n\nThe majority of published data on the food habits of bats are based on analyses of stomach and fecal contents. Stomach contents analyses allow examination of the undigested items of the last meal, whereas fecal analysis is biased towards detection of less digestible items because digestion appears to degrade soft parts quickly (Whitaker, 1988). Therefore, to obtain the best results, the stomach content analysis method was used in this study. According to our diet composition results despite the different habitats from which the samples were collected, they did not significantly affect the type of insects that were identified in the stomach contents at the level of insect order. There may have been differences at the level of the insect species within the same order, this may be due to the overlap of these habitats and their close proximity to each other, or it may be due to the nature of Kuhl’s bats and their ability to forage over variety of habitats as open natural land and agricultural fields, above water ponds, at vegetation edge and attracted to insects on street lights in rural and urban areas (Kalko and Schnitzler 1993; Ancillotto et al. 2015, 2016; Kahnonitch et al. 2018). Kuhl’s bat has been considered as a generalist insectivore, with high behavioral flexibility, may utilize variety of environments (Kalko and Schnitzler 1993), whereas it doesn’t show prefer a specific insects or feeding habitat (Goiti et al. 2003).According to our finding, prey species belonging to the insect order Hymenoptera had the highest frequency of occurrence and are thus on the top of the food spectrum of the Pipistrellus kuhlii bats, followed by Lepidoptera and Coleoptera sequentially. This insects orders were consistent with available information on the diet of Pipistrellus bats, including Coleoptera, Diptera, Hemiptera, Heteroptera, Hymenoptera, and Lepidoptera were reported after (Whitaker et al. 1994; Feldman et al. 2000; Goiti et al. 2003; Benda et al. 2006, 2010; Cohen et al. 2020). As previous studies indicate that, the variation in the volumetric representation of insect orders were preyed by this bat depending on the variation of the geographical location and the seasons, Whitker et al. (1994), found that Pipistrellus kuhlii in one location fed mainly on hymenopterans and it’s make up 56 %, followed by Coleoptera (32 %) and Lepidoptera (6 %). While in another location Diptera was a dominant group (31 %), followed by Lepidoptera (24 %), Coleoptera (17 %), and Hymenoptera (15 %) (Feldman et al. 2000). In Syria Benda et al. (2006) found that Hemiptera was a dominant group (35 %), followed by Coleoptera (21 %), Hymenoptera (14 %), and Lepidoptera (11 %). The variability of the frequency of some insect species belonging to one order compared to another may be due to the generalist behavior and high behavioral adaptability, of this bat species. So, the more likely reason is that it is due to the availability and abundance of these prey orders in foraging areas, Whitaker and Karatas\u0001 (2009) concluded that Pipistrellus bats feeding on a variety of insect species and can be treated as generalists. Fruits and unknown materials were also found to be present in the stomachs of this bat species.\n\nWhile Pipistrellus kuhlii bats are historically known for their feeding habits that focus on a small number of insect orders, consumed at high frequency and volume in their diet, our findings show that they feed on a variety of insect orders. As a result, insects are an essential source of energy for bats. Pipistrellus kuhlii bats feed mainly in the first few hours after sunset, and their feeding behavior is characterized by low, quick feeding flights in open areas (Barak and Yom-Tov 1989). According to Serangeli et al. (2012) and Maxinová et al. (2016) the home range of Kuhl’s bat is less than 2 km<sup>2</sup> but the foraging sites as far as 4.5 km.\n\nWhile fruit remains occurred in the diet, they were not represented by a high frequency of occurrence or volume in the stomach and remains were restricted to three individuals which collected from the urban areas. This suggests that these were consumed accidentally by bats or alternatively it could be a result of an adapted feature by these insectivorous bats as a result of living in urban areas. Bats might forage on insects flying or attached to fruit remains in trash. For example, this bat was attracted to insects swarming around ripe fruits on a peach tree in East Africa, according to Kingdon (1974), which may explain the presence of fruits remains in their diet.\n\n## 5. Conclusion\n\nThis study also establishes a baseline for the diet of this insectivorous bat species across a variety of habitats in the Nile Valley. Our findings of the stomach contents, despite the different habitats from which the samples were collected, didn’t significantly affect the type of insects that were identified at the level of insect order. Still, there may have been differences at the level of the insect species within the same order. The three most common insect taxa recovered were Hymenoptera, Lepidoptera and Coleoptera. The percentage volume of insect orders in the diet of Kuhl’s bat (63 stomachs) consisted of 29 % of Hymenoptera, 9 % of Coleoptera,\n\n27 % of Lepidoptera, 32 % unknown materials and unexpected thing of these insectivorous bats was the presences of 3 % fruits. A single order did not dominate the diet of this specie because Hymenoptera was the most fed insect orders by frequency of occurrence 87.3 %, Lepidoptera by 74.6 %, Coleoptera 36.5 % and fruits in 4.8 % of samples. Also, they may be fed by other insect orders, as unknown materials were found in all samples.\n\n## Declaration of Competing Interest\n\nThe authors declare that they have no known competing financial interests or personal relationships that could have appeared to influence the work reported in this paper.\n\n## Acknowledgment:\n\nWe thanks Dr. Sarah Davies. Ph.D., school of Biosciences, Cardiff University, United Kingdom, for valuable help in revising the English language of the Manuscript as a native English speaker to edit the text.\n\n## References\n\n[Amichai, E., 2017. Flexibility and plasticity in active sensing in echolocating bats. Tel-Aviv University. Doctoral dissertation](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0005).\n\n[Amichai, E., Korine, C., 2020. Kuhl’s pipistrelle Pipistrellus kuhlii (Kuhl, 1817). Handbook of the Mammals of Europe, 1–19](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0010).\n\n[Ancillotto, L., Tomassini, A., Russo, D., 2015. The fancy city life: Kuhl’s pipistrelle, Pipistrellus kuhlii, benefits from urbanization. Wildlife Research 42 (7), 598–606](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0015).\n\n[Ancillotto, L., Santini, L., Ranc, N., Maiorano, L., Russo, D., 2016. Extraordinary range expansion in a common bat: the potential roles of climate change and urbanization. The Science of Nature 103 (3), 1–8](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0020).\n\n[Barak, Y., Yom-Tov, Y., 1989. The advantage of group hunting in Kuhl’s bat Pipistrellus kuhli (Microchiroptera). J. Zool. 219 (4), 670–675](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0025).\n\nBar-David, S., Shahar, N., Kopelman, N., Dolev, A., Levin, E., Amichai, E., Korine, C., 2015. The range expantion of Kuhl’s pipistrelle bat in Israel: what can be learnt from the population’s genetics? In Proceedings of the 27th international congress for conservation biology and 4th European congress for conservation biology. Montpellier, pp 45–46.\n\nBenda, P., Andreas, M., Kock, D., Lučan, R.K., Munclinger, P., Nová, P., Obuch, J., [Ochman, K., Reiter, A., Uhrin, M., Weinfurtová, D., 2006. Bats (Mammalia: Chiroptera) of the Eastern Mediterranean. Part 4. Bat fauna of Syria: distribution, systematics, ecology. Acta Societatis Zoologicae Bohemicae 70, 1–329](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0035).\n\nBenda, P., Dietz, C., Andreas, M., Hotový , J., Lučan, R.K., Maltby, A., Vallo, P., 2008. [Bats (Mammalia: Chiroptera) of the Eastern Mediterranean and Middle East. Part 6. Bats of Sinai (Egypt) with some taxonomic, ecological and echolocation data on that fauna. Acta Societatis Zoologicae Bohemicae 72 (1–2), 1–103](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0040).\n\n[Benda, P., Lucan, R.K., Obuch, J., Reiter, A., Andreas, M., Backor, P., Amr, Z.S., 2010. Bats (Mammalia: Chiroptera) of the Eastern Mediterranean and Middle East. Part 8. Bats of Jordan: fauna, ecology, echolocation, ectoparasites. Acta Societatis Zoologicae Bohemicae 74 (3–4), 185–353](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0045).\n\n[Borror, D.J., Triplehorn, C.A., Borror, N.F., 1989. An introduction to the study of insects. Sanders College Publishing, Orlando, Fla](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0050).\n\n[Bray, T.C., Mohammed, O.B., Alagaili, A.N., 2013. Phylogenetic and demographic insights into Kuhl’s Pipistrelle, Pipistrellus kuhlii, in the Middle East. PLoS ONE 8 (2), e57306](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0055).\n\n[Carroll, S.K., Carter, T.C., Feldhamer, G.A., 2002. Placement of nets for bats: effects on perceived fauna. Southeast. Nat. 1, 193–198](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0060).\n\n[Cohen, Y., Bar-David, S., Nielsen, M., Bohmann, K., Korine, C., 2020. An appetite for pests: Synanthropic insectivorous bats exploit cotton pest irruptions and consume various deleterious arthropods. Mol. Ecol. 29 (6), 1185–1198](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0065).\n\n[Dalhoumi, R., Morellet, N., Aissa, P., Aulagnier, S., 2018. Seasonal activity pattern and habitat use by the Kuhl’s pipistrelle (Pipistrellus kuhlii) in an arid environment. Eur. J. Wildl. Res. 64 (3), 1–12](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0070).\n\n[Dietz, C., 2005. Illustrated identification key to the bats of Egypt. Electronic publication version, 1](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0075).\n\nDietz, C., and von Helversen, O. 2004. Illustrated identification key to the bats of Europe.\n\n[Feldhamer, G.A., Carter, T.C., Carroll, S.K., 2001. Timing of pregnancy, lactation, and female foraging activity in three species of bats in southern Illinois. Can. Field-Nat. 115, 420–424](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0085).\n\n[Feldman, R., Whitaker, J.O., Yom-Tov, Y., 2000. Dietary composition and habitat use in a desert insectivorous bat community in Israel. Acta Chiropterologica 1 (02), 15–22](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0090).\n\n[Goiti, U., Vecin, P., Garin, I., Saloña, M., Aihartza, J.R., 2003. Diet and prey selection in Kuhl’s pipistrelle Pipistrellus kuhlii (Chiroptera: Vespertilionidae) in southwestern Europe. Acta Theriologica 48 (4), 457–468](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0095).\n\n[Harrison, D., Bates, P., 1991. The Mammals of Arabia. Harrison Zoological Museum Publ, Sevenoaks, Kent, p. 354](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0100).\n\nKahnonitch, I., Lubin, Y., Korine, C., 2018. Insectivorous bats in semi-arid agro ecosystems effects on foraging activity and implications for insect pest control. Agaric Ecosystem Environ 261, 80–92. [https://doi.org/10.1016/j.agee.2017.11.003](https://doi.org/10.1016/j.agee.2017.11.003).\n\n[Kalko, E.K., Schnitzler, H.U., 1993. Plasticity in echolocation signals of European pipistrelle bats in search flight: implications for habitat use and prey detection. Behav. Ecol. Sociobiol. 33 (6), 415–428](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0110).\n\n[Karanth, K.U., Sunquist, M.E., 1995. Prey selection by tiger, leopard and dhole in tropical forests. J. Anim. Ecol. 64, 439–450](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0115).\n\n[Kingdon, J., 1974. East African Mammals: insectivores and bats, Vol. IIA. University of Chicago Press, Chicago](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0120).\n\n[Maxinová, E., Kipson, M., Nad’o, L., Hradická, P., Uhrin, M., 2016. Foraging strategy of Kuhl’s pipistrelle at the northern edge of the species distribution. Acta Chiropterologica 18 (1), 215–222](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0125).\n\n[Mcaney, K., Shiel, C., Sullivan, C., Fairley, J., 1991. The Analysis of Bat Droppings. Occasional publication of the Mammal Society, London 14, 1–48](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0130).\n\n[Mendelssohn, H., Yom-Tov, Y., 1999. Fauna Palestina. The Israel Academy of Sciences and Humanities, Mammalia of Israel. Jerusalem, p. 439](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0135).\n\n[Nunez, R., Miller, B., Lindzey, F., 2000. Food habits of jaguars and pumas in Jalisco, Mexico. Journal of Zoology (London) 252, 373–379](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0140).\n\n[Osborn, D.J., Helmy, I., 1980. The contemporary land mammals of Egypt (including Sinai). Field Museum of Natural History, Chicago ILL](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0145).\n\n[Qumsiyeh, M.B., 1985. The bats of Egypt. Special Publication of the Museum of Texas Tech University 23, 1–101](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0150).\n\n[Russo, D., Ancillotto, L., 2015. Sensitivity of bats to urbanization: a review. Mamm. Biol. 80 (3), 205–212](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0155).\n\n[Serangeli, M.T., Cistrone, L., Ancillotto, L., Tomassini, A., Russo, D., 2012. The post-release fate of hand-reared orphaned bats: survival and habitat selection. Animal Welfare-The UFAW Journal 21 (1), 9](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0160).\n\n[Sharifi, M., Ghorbani, R., Akmali, V., 2004. Reproductive cycle in Pipistrellus kuhlii (Chiroptera, Vespertilionidae) in western Iran. Mammalia mamm 68, 323](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0165).\n\n[Taber, A.B., Novaro, A.J., Neris, N., Colman, F.H., 1997. The food habits of sympatric jaguar and puma in the Paraguayan Chaco. Biotropica 29, 204–213](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0170).\n\n[United States Fish Andand Wildlife Service, 1999. Agency draft Indiana bat (Myotis sodalis) revised recovery plan. United States Fish and Wildlife Service, Ft. Snelling, Minnesota. Region 3,](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0175).\n\n[Wassif, K., 1995. Guide to Mammals of Natural Protectorates in Egypt. Publications of National Biodiversity Unit. No.4, Cairo, p. 171](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0180).\n\nWaWrocka, K., Bartonička, T., Reiter, A., 2012. Pipistrellus kuhlii, a bat species [breeding and hibernating in the Czech Republic. Vespertilio 16, 351–356](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0185).\n\n[Whitaker, J.O., 1988. Food habits analysis of insectivorous bats. In: Kunz, T.H. (Ed.), Ecological and Behavioral Methods for the Study of Bats. Smithsonian Inst. Press, Washington, D. C, pp. 171–189](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0190).\n\n[Whitaker, J.O., 2004. Prey selection in a temperate zone insectivorous bat community. J. Mammal. 85, 460–469](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0195).\n\n[Whitaker, J.O., Karata](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0200)s\u0001[, A., 2009. Food and feeding habits of some bats from Turkey. Acta Chiropterologica 11 (2), 393–403](http://refhub.elsevier.com/S1319-562X%2822%2900406-5/h0200).\n\nWhitker, j. O., Shalmon, B., Kunz, T. H., 1994. Food and feeding habits of insectivorous bats from Israel. Z. Saugetierkunde, 59, 74-81.",
    "output": "[['references', 'verbatimScientificName', 'taxonRank', 'verbatimTraitName', 'verbatimTraitUnit', 'individualCount', 'measurementValue_min', 'measurementValue_max', 'dispersion', 'statisticalMethod', 'verbatimTraitValue', 'sex', 'lifeStage', 'measurementMethod', 'measurementRemarks', 'measurementAccuracy', 'measurementDeterminedBy', 'verbatimLocality', 'author', 'associatedReferences'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'BM = Body Mass in grams (g)', 'g ', '', '4.9', '5.8', '', '', '5.2', 'male', '', '', '', '', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'BM = Body Mass in grams (g)', 'g ', '', '4.5', '5.4', '', '', '4.8', 'female', '', '', '', '', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'TBL = Total Body Length', 'mm ', '', '152', '175', '', '', '166.3', 'male', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'WS = Wing Span (width of the bat)', 'mm ', '', '152', '175', '', '', '166.3', 'male', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'WS = Wing Span (width of the bat)', 'mm ', '', '118', '168', '', '', '149.5', 'female', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'FA = Forearm length', 'mm', '', '32', '34.6', '', '', '33.2', 'male', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'FA = Forearm length', 'mm', '', '32.4', '33.3', '', '', '32.9', 'female', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'D5 = 5th Digit length', 'mm', '', '40', '44.5', '', '', '41.9', 'male', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'D5 = 5th Digit length', 'mm ', '', '40', '41.4', '', '', '40.5', 'female', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'D3 = 3rd Digit length', 'mm ', '', '54', '58.6', '', '', '56.2', 'male', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'D3 = 3rd Digit length', 'mm ', '', '54.1', '57.2', '', '', '55.1', 'female', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'TL = Tail Length', 'mm ', '', '27.5', '36.4', '', '', '30.7', 'male', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'TL = Tail Length', 'mm ', '', '21.4', '31.5', '', '', '27.8', 'female', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'TBL = Total Body Length', 'mm ', '63', '60.3', '76.4', '', '', '', '', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'TL = Tail Length', 'mm ', '63', '21.4', '36.4', '', '', '', '', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'WS = Wing Span (width of the bat)', 'mm ', '63', '118', '175', '', '', '', '', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'FA = Forearm length', 'mm ', '63', '32', '34.6', '', '', '', '', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'D5 = 5th Digit length', 'mm ', '63', '40', '44.5', '', '', '', '', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'D3 = 3rd Digit length', 'mm ', '63', '54', '61', '', '', '', '', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'BM = Body Mass in grams (g)', 'g ', '63', '4.5', '5.8', '', '', '', '', '', '', '', '', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'EL = Ear Length', 'mm ', '63', '8.2', '9.6', '', '', '', '', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study']]"
  },
  {
    "instruction": "\nYou're an expert text parser. Your goal is to parse TSV fields from the input text. A single input text file can have multiple \ndifferent species and/or multiple individuals from the same species. The output should include a TSV header and the following\nfields should be filled for each species with information found from the input text:\n\nreferences:\nDescription: Complete bibliographic reference formatted according to Harvard citation style.\nExamples: Smith, J., Doe, A. and Jones, B. (2020) 'Plant functional traits across biomes', Journal of Ecology, 108(4), pp. 1200-1215.\n\nverbatimScientificName:\nDescription: The full, original scientific name as supplied in the raw data, including authorship or subspecific rank if recorded.\nExamples: Pinus sylvestris L., Panthera leo (Linnaeus, 1758)\n\ntaxonRank:\nDescription: Taxonomic rank of the target organism.\nExamples: species, genus, subspecies\n\nverbatimTraitName:\nDescription: Exact trait name as originally recorded in the source data or publication.\nExamples: SLA, specific leaf area, body mass\n\nverbatimTraitUnit:\nDescription: Original unit of measurement as stated in the source.\nExamples: mm, g, mm2/mg, kg\n\nindividualCount:\nDescription: Number of individuals aggregated into the reported measurement.\nExamples: 1, 15, 100\n\nmeasurementValue_min:\nDescription: Minimum measured value within a range or aggregate group.\nExamples: 12.4, 0.5\n\nmeasurementValue_max:\nDescription: Maximum measured value within a range or aggregate group.\nExamples: 28.1, 3.2\n\ndispersion:\nDescription: Measure of variation/dispersion associated with the central value (e.g., standard deviation, standard error).\nExamples: SD = 1.2, 0.45\n\nstatisticalMethod:\nDescription: Statistical metric used to summarize the values.\nExamples: mean, median, mode, range\n\nverbatimTraitValue:\nDescription: Raw value or text string as originally reported (numerical or categorical).\nExamples: 15.4, herbaceous, red\n\nsex:\nDescription: Biological sex of the specimen(s).\nExamples: male, female, hermaphrodite, mixed\n\nlifeStage:\nDescription: Developmental age or life stage of the organism.\nExamples: adult, juvenile, seedling, larva\n\nmeasurementMethod:\nDescription: Protocols, instruments, or procedures used to capture the measurement.\nExamples: Swinbank dry weight method, Caliper measurement of tarsus length\n\nmeasurementRemarks:\nDescription: Free-text notes or qualifying details about the measurement process or state of the specimen.\nExamples: Measured on dried museum specimens, Leaves collected from upper canopy\n\nmeasurementAccuracy:\nDescription: Precision, error tolerance, or accuracy rating of the instrument/method used.\nExamples: ± 0.01 g, 0.1 mm\n\nmeasurementDeterminedBy:\nDescription: Name or ORCID of the person/entity who made or validated the measurement.\nExamples: Jane Doe, https://orcid.org/0000-0000-0000-0000\n\nverbatimLocality:\nDescription: Original description of the sampling location before standardized georeferencing.\nExamples: Mount Rainier National Park, trail to Paradise, Plot 4B, Black Forest\n\nauthor:\nDescription: This field has to only include bioparser\nExamples: BioParser, BioParser, BioParser\n\nassociatedReferences:\nDescription: Citations, DOIs, or bibliographic references supporting the dataset or measurement.\nExamples: https://doi.org/10.1111/1365-2435.12345, Smith et al. 2020\n\nFULL OUTPUT EXAMPLE:\n[['references', 'verbatimScientificName', 'taxonRank', 'verbatimTraitName', 'verbatimTraitUnit', 'individualCount', 'measurementValue_min', 'measurementValue_max', 'dispersion', 'statisticalMethod', 'verbatimTraitValue', 'sex', 'lifeStage', 'measurementMethod', 'measurementRemarks', 'measurementAccuracy', 'measurementDeterminedBy', 'verbatimLocality', 'author', 'associatedReferences'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'BM = Body Mass in grams (g)', 'g ', '', '4.9', '5.8', '', '', '5.2', 'male', '', '', '', '', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'BM = Body Mass in grams (g)', 'g ', '', '4.5', '5.4', '', '', '4.8', 'female', '', '', '', '', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'TBL = Total Body Length', 'mm ', '', '152', '175', '', '', '166.3', 'male', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'WS = Wing Span (width of the bat)', 'mm ', '', '152', '175', '', '', '166.3', 'male', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'WS = Wing Span (width of the bat)', 'mm ', '', '118', '168', '', '', '149.5', 'female', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'FA = Forearm length', 'mm', '', '32', '34.6', '', '', '33.2', 'male', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'FA = Forearm length', 'mm', '', '32.4', '33.3', '', '', '32.9', 'female', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'D5 = 5th Digit length', 'mm', '', '40', '44.5', '', '', '41.9', 'male', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'D5 = 5th Digit length', 'mm ', '', '40', '41.4', '', '', '40.5', 'female', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'D3 = 3rd Digit length', 'mm ', '', '54', '58.6', '', '', '56.2', 'male', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'D3 = 3rd Digit length', 'mm ', '', '54.1', '57.2', '', '', '55.1', 'female', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'TL = Tail Length', 'mm ', '', '27.5', '36.4', '', '', '30.7', 'male', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'TL = Tail Length', 'mm ', '', '21.4', '31.5', '', '', '27.8', 'female', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'TBL = Total Body Length', 'mm ', '63', '60.3', '76.4', '', '', '', '', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'TL = Tail Length', 'mm ', '63', '21.4', '36.4', '', '', '', '', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'WS = Wing Span (width of the bat)', 'mm ', '63', '118', '175', '', '', '', '', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'FA = Forearm length', 'mm ', '63', '32', '34.6', '', '', '', '', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'D5 = 5th Digit length', 'mm ', '63', '40', '44.5', '', '', '', '', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'D3 = 3rd Digit length', 'mm ', '63', '54', '61', '', '', '', '', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'BM = Body Mass in grams (g)', 'g ', '63', '4.5', '5.8', '', '', '', '', '', '', '', '', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'EL = Ear Length', 'mm ', '63', '8.2', '9.6', '', '', '', '', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study']]\"\n",
    "input": "# Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia\n\nMounir R. Abi-Said<sup>1,\\*</sup> and Mohammad S. Al-Zein<sup>2</sup>\n\n<sup>1</sup>Laboratory of Geosciences, Georesources, and Environment, Department of Life and Earth Sciences, Faculty of Science II, Lebanese University. PO Box 90656 Jdeideh, Fanar; <sup>2</sup>Department of Biology, Faculty of Arts & Sciences, American University of Beirut, Riad El Solh 1107 2020 Beirut Lebanon.\n\nReceived: June 2, 2022; Revised June 25, 2022; Accepted: June 30, 2022\n\n**Abstract:** The rodents of Ar Riyadh Province, Kingdom of Saudi Arabia are not well documented. Twenty Sheman live traps were distributed in 23 sites along Wadi As Sulai in Ar Riyadh Province between February and April 2016 as a part of an ecological survey for the area. Nine species of rodents belonging to two families were caught four of which were reported for the first time at the species level in Ar Riyadh Province namely, Gerbillus dasyurus, Gerbilus cheesmani, Gerbilus nanus, and Meriones crassus. External, cranial, and dental measurements were recorded.\n\n**Keywords**: Rodents, Dipodidae, Muridae, Wadi As Sulai, Riyadh, Saudi Arabia.\n\n## Introduction\n\nRodents are considered as pests and vectors of many zoonotic diseases. Nonetheless, rodents play important ecological roles that are vital for the sustainability of ecosystems. Rodents are keystone species in many ecosystems controlling the ecosystem’s structure and affecting vertebrate and invertebrate species richness (Brown and Heske, 1990; Delibes-Mateos et al., 2011). They constitute an important prey base for a large number of predators in addition to being seed disseminators and forest regenerators (Orrock et al., 2004; Howe and Brown, 2000). Moreover, rodents play important ecological roles in desert ecosystems (Reichman, 1979). Hence studying rodent distribution and their existence in an ecosystem will lead to better understanding of the ecosystem.\n\nFew studies on rodents have been conducted <u>in Saudi Arabia (e.g. Harrison,</u> 1972;\n\nButtiker and Harisson, 1982; Al-Rajhi et al., 1993; Al-Ahmed and Al-Dawood, 2001; López-Antoñanzas and Sen, 2006; Strauss et al., 2008; Henry and Dubost, 2012; Sarli et al., 2016). Harrison and Bates (1991) reported 46 rodent species (excluding the Sciuridae and the Hystricidae) in the Arabian Peninsula, twenty-two species of which are present in Saudi Arabia. According to the first Saudi Arabian National Report on the Convention on Biological Diversity, 22 species and subspecies of rodents belonging to fourteen genera and five families exist in Saudi Arabia and five of which are endemic to Saudi Arabia (Gerbillus cheesmani arduus; Meriones rex philbyi; Meriones crassus longifrons; Meriones libycus arimalius; Psammomys obesus dianae) (Abu Zinada et al., 2003). Lewis et al. (1963) reported eight rodent species from Northern Saudi Arabia, Al Kalili (1984) documented five species of rodents from the southwest of Saudi Arabia. Very few assessments examined rodents in Ar-Riyadh Province. Four species belonging to two families were reported from Ar Riyadh city (Al-Rajhi et al 1993) and Al-Ahmed and Al-Dawood (2002) collected six species from Wadi Hanifah, which is parallel to Wadi As-Sulai in Riyadh city where the current study was conducted.\n\nWadi As-Sulai is a new project that surrounds Ar-Riyadh city parallel to Wadi Hanifa. It represents a man-made drainage system for rainwater drainage around Riyadh city. No studies on rodents were conducted on this site. Hence Rodents were studied in this area as a part of an environmental impact assessment study for the Wadi. Most recently, Abi-Said et al (2020) reported on the presence of at least 11 species of rodents based on remains from pellets of the Pharaoh Eagle Owl Bubo ascalaphus. This study aims to augment the information on rodents in Ar-Riyadh Province and to highlight the effect of the drainage system on their population.\n\n<small><span class=\"docvortex-page-footnote\" data-block-type=\"page_footnote\" style=\"color:#6b7280\">\\*Corresponding author: mabisaid9@gmail.com</span></small>\n\n## Material and Methods\n\n## Area description\n\nThe area of Wadi As-Sulai is a representative of the desert ecosystem sheltering typical desert life forms. It is part of the Kharj drainage basin, is an ephemeral waterway that collects runoff water from more elevated areas surrounding it. Water draining into the wadi comes from the natural watershed (2,550 km<sup>2</sup>), which includes Eastern Riyadh, with additional inputs from water treatment plants; the water is ultimately lost in the desert. Wadi As-Sulai stretches for ca. 120 km from north to south, its width sometimes only a few meters. It is comprised of habitats that range from semi-natural to widely disturbed, the major disturbances being grazing, pollution (discharge of partially treated and crude wastewater, construction wastes, litter, etc.), urbanization, and offroad driving. Fencing in the airport area as well as the National Guard area has partially excluded people and their animals; thus, these areas are more densely vegetated than adjacent areas. Apart from seasonal water bodies, the Wadi includes some sand/ silt pans in addition to semi-permanent and permanent water bodies, some filled with treated or partially treated wastewater.\n\n## Rodent trapping\n\nThe trapping stretched from the February 3<sup>rd</sup> to April 12<sup>th</sup>, 2016. Twenty-three sites covering all the Wadi were selected for rodent trapping using 20 Sherman live traps for one night per site (Figure 1).\n\nDue to the limited number of traps, three stations were selected per night. In each station, twenty traps baited with peanut butter, oat, sunflower seeds, and canary feed-mix were set at dusk and were checked the next day at dawn. Traps were arranged in line transects close to burrows. The caught animals were identified according to Harrison and Bates (1991), measured, photographed, skinned and skulls were cleaned and measured. All animals were taken as specimens and kept at the Department of Life and Earth Sciences collection, Lebanese University.\n\n## Results\n\nA total of 460 trapping nights resulted in trapping 62 individuals belonging to nine species belonging to two families (Table 1, Figure 2). The conservation status of all trapped species were of least concern (LC) according to IUCN redlist 2016. The most trapped rodent was Acomys dimidiatus (30.65%) followed by Meriones libycus (27.42%) and the Gerbillus spp. were the least trapped (Table 1). A. dimidiatus were caught in rocky areas while the M. libycus were caught in sandy areas while Rattus rattus and Mus musculus around urban areas.\n\nTable 1. List of rodent species and their trapping percentage at Wadi As Sulai\n\n| Family | Scientific Name | N | % |\n| --- | --- | --- | --- |\n| Dipodidae | Jaculus jaculus | 6 | 9.68 |\n| Muridae | Acomys dimidiatus | 19 | 30.65 |\n|  | Gerbillus cheesmani | 3 | 4.84 |\n|  | Gerbillus dasyurus | 1 | 1.61 |\n|  | Gerbillus nanus | 2 | 3.22 |\n|  | Meriones crassus | 8 | 12.90 |\n|  | Meriones libycus | 17 | 27.42 |\n|  | Mus musculus | 2 | 3.23 |\n|  | Rattus rattus | 4 | 6.45 |\n\nExternal, cranial, and dental measurements of species caught were recorded (Tables 2 and 3).\n\n## Discussion\n\nDuring this rapid survey, nine species of rodents were identified (Table 1). Al-Ahmed and Al-Dawood (2002) identified six rodent species in Wadi Hanifah which is parallel to\n\n![](images/page_2_image_2.jpg)\n\nFigure 1. Trapping sites along Wadi As Sulai\n\nTable 2. External measurements of the rodents trapped in Wadi As Sulai\n\n<table><tbody><tr><td rowspan=\"2\">Species</td><td rowspan=\"2\">N</td><td rowspan=\"2\">Wt. (g)</td><td rowspan=\"2\">HB (mm)</td><td rowspan=\"2\">TL (mm)</td><td rowspan=\"2\">HL (mm)</td><td rowspan=\"2\">HW (mm)</td><td rowspan=\"2\">E (mm)</td><td rowspan=\"2\">HF (mm)</td></tr><tr></tr><tr><td>A. dimidiatus</td><td>19</td><td>41.22</td><td>109.49</td><td>96.52</td><td>33.46</td><td>15.51</td><td>17.25</td><td>20.45</td></tr><tr><td>D. dasyrus</td><td>1</td><td>18</td><td>84.12</td><td>114.43</td><td>27.77</td><td>14.52</td><td>12.5</td><td>20.52</td></tr><tr><td>G. cheesmani</td><td>3</td><td>23.5</td><td>89.22</td><td>116.45</td><td>29.59</td><td>16.04</td><td>12.01</td><td>25.675</td></tr><tr><td>G. nanus</td><td>2</td><td>14.5</td><td>77.12</td><td>103.75</td><td>27.5</td><td>14.45</td><td>9.01</td><td>23.54</td></tr><tr><td>J. jaculus</td><td>6</td><td>48.5</td><td>114.21</td><td>168.16</td><td>34.07</td><td>25.79</td><td>19.66</td><td>60.72</td></tr><tr><td>M. libycus</td><td>17</td><td>146.1</td><td>167.2</td><td>156.5</td><td>44.1</td><td>23.5</td><td>17.3</td><td>41.38</td></tr><tr><td>M. crassus</td><td>8</td><td>80.33</td><td>126.88</td><td>138.73</td><td>38.13</td><td>21.29</td><td>14.01</td><td>33.78</td></tr><tr><td>M. musculus</td><td>2</td><td>7.6</td><td>65.66</td><td>69.85</td><td>21.21</td><td>10.42</td><td>12.72</td><td>18.51</td></tr><tr><td>R. rattus</td><td>4</td><td>161</td><td>184.14</td><td>185.84</td><td>39.98</td><td>21.71</td><td>24.93</td><td>35.45</td></tr></tbody></table>\n\nHB: Head and body, TL: Tail; HF: Hindfoot; HL: Head length, HW: Head width, E: Ear; Wt. g: wight in grams\n\nTable 3. Cranial and dental measurements of rodents trapped in Wadi As Sulai\n\n| Species | N | IC | ZB | BB | GTL | CBL | MXC | MDC | M |\n| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |\n| A. dimidiatus | 19 | 4.93 | 13.71 | 11.2 | 30.41 | 28.83 | 4.79 | 4.47 | 18.25 |\n| D. dasyrus | 1 | 5.23 | 11.03 | 12.73 | 26.42 | 23.32 | 3.79 | 3.13 | 14.22 |\n| G. cheesmani | 3 | 5.2 | 12.67 | 9.67 | 21.42 | 19.24 | 3.72 | 3.72 | 15.34 |\n| G. nanus | 2 | 5.04 | 11.57 | 12.83 | 28.9 | 25.8 | 3.98 | 3.75 | 12.48 |\n| J. jaculus | 6 | 11.01 | 20.61 | 22.5 | 31.9 | 28.1 | 4.65 | 4.46 | 18.7 |\n| M. crassus | 8 | 5.86 | 15.34 | 13.23 | 30.86 | 28.53 | 5.24 | 5.12 | 19.06 |\n| M. libycus | 17 | 6.85 | 17.4 | 14.61 | 36.82 | 34.43 | 5.54 | 6.57 | 20.36 |\n| M. musculus | 2 | 3.59 | 9.01 | 7.94 | NA | 19.97 | 3.16 | 2.77 | 11.28 |\n| R. rattus | 4 | 5.95 | 18.37 | 13.52 | 41.11 | 38.92 | 6.55 | 6.16 | 24.46 |\n\nGtL: Greatest length of skull; ZB: Zygomatic breadth; IC: Interorbital constriction; BB: Brain case breadth; MXC: Maxillary cheekteeth; MDC: Mandibular cheekteeth; M: Mandible length (incisor included).\n\nWadi As Sulai these included A. dimidiatus, M. libycus, M. musculus and three subspecies of R. rattus. While El-Bahrawy and Al-Dakhil (1993) trapped seven rodent species in Ar Riyadh namely; J. jaculus, A. dimidiatus, M. musculus, R. rattus, R. norvegicus, and one Meriones spp, and one Gerbillus spp. Al-Rajhi et al. (1993) collected only four species A. dimidiatus, M. libycus, M. musculus, and R. rattus. This high number of species caught is probably due to the suitable season for rodent activity where a favorable climate prevails compared to the other study in Wadi Hanifah that was carried on between August and November when the weather is very hot. The high percentage of A. dimidiatus is owed to the suitable landscape of rocky structures where this species can hide easily. Similarly, the high percentage of M. libycus caught is due to the spread of farmlands within the study site. Similarly, Al-Ahmed and Al-Dawood (2002) trapped more M. libycus in farmlands in Wadi Hanifah is also similar to the results reported by El-Bahrawy and Al-Dakhil (1993), Al-Rajhi et al. (1993) and Abi-Said et al. (2020).\n\nMoreover, external, cranial, and dental measurements of species caught were similar to those reported by Harrison and Bates (1991) and Aulagnier et al. (2009). This study augments the rodent fauna in Ar Riyadh Province and highlights the importance of timing the trapping activity to match the activity of the rodents in the study site. This rapid survey identified four rodents that were not reported to the species level including;\n\n![](images/page_4_image_2.jpg)\n\nFigure 2: A. The three-toed jarboa, Jaculus jaculus. B. Arabian spiny mouse, Acomys dimidiatus. C. Cheesmani gerbil, Gerbillus cheesmani. D. Wagner’s gerbil, Gerbillus dasyurus. E. The Libyan jird, Meriones libycus. F. The black rat, Rattus rattus.\n\nG. cheesmani, G. dasyrus, G. nanus, and M. crassus, and provided additional external and cranial measurements. Hence a comprehensive study to cover the entire Ar Riyadh Province is needed to document the small mammals present and their relative abundance.\n\n## References\n\nAbi-Said, MR, Al Zein, MS, Amr, ZS and Abu Baker, MA. 2020. Diet of the desert eagle owl, Bubo ascalaphus, in eastern Saudi Arabia. Pakistan Journal of Zoology, 52: 1169–1171.\n\nAbu Zinada, A H, Robinson, E R, Nader, I A and Al Wetaid, Y I. 2003. First\n\nSaudi Arabian National Report on the Convention on Biological Diversity. The National Commission for Wildlife Conservation and Development, Riyadh.\n\nAl-Ahmed, A M and Al-Dawood, AS. 2001. Rodents and their ectoparasites in Wadi Hanifah, Riyadh City, Saudi Arabia. Journal Egyptian Society of Parasitology, 31:737-744.\n\nAl-Khalili, A D. 1984. New records of ectoparasites of small rodents from SW Saudi Arabia. Fauna of Saudi Arabia, 6: 510–512.\n\nAl-Rajhi, D H, Tag-EL-Din, A, Hussein, H I and Mostafa, S. 1993. Trapping of Rodent Pests in Riyadh Region, Saudi Arabia. Agricultural Research Bulletin. 36 pp.\n\nAulagnier, S, Haffner, P, Mitchell-Jones, AJ, Moutou, F, and Zima, J. 2009. Mammals of Europe, North Africa and the Middle East. A & C Black Publishers Ltd., London, UK.\n\nBrown, J H and Heske, E J. 1990. Control of a desert-grassland transition by a keystone rodent guild. Science (Washington), 250 (4988): 1705-1707.\n\nButtiker, W and Harrison, DL. 1982. Mammals of Saudi Arabia. On a collection of Rodents from Saudi Arabia. Fauna of Saudi Arabia, 4:488-502.\n\nDelibes-Mateos, M, Smith, A T, Slobodchikoff, C N and Swenson, J E. 2011. The paradox of keystone species persecuted as pests: a call for the conservation of abundant small mammals in their native range. Biological Conservation, 144: 1335-1346.\n\nEl Bahrawy, AA and Al Dakhil, MA. 1993. Studies on the ectoparasites (fleas and lice) on rodents in Riyadh and its surroundings, Saudi Arabia. Journal of the Egyptian Society of Parasitology, 23:723-35.\n\nHarrison, DL. 1972. The Mammals of Arabia. 3. Lagomorpha and Rodentia. Benn, London. pp. 382–670.\n\nHarrison, DL and Bates, PJJ. 1991. The Mammals of Arabia. Sevenoaks: Harrison Zoological Museum.\n\nHenry, O and Dubost, G. 2012. Breeding periods of Gerbillus cheesmani (Rodentia, Muridae) in Saudi Arabia. Mammalia, 76: 383-387.\n\nHowe, HF and Brown, JS. 2000. Early effects of rodent granivory on experimental forb communities. Ecological Applications, 10: 917-924.\n\nIUCN Red List of Threatened Species 2016: e.T136471A115208221. https://dx.doi.org/10.2305/IUCN.UK.2016-3.RLTS.T136471A22453198.en.\n\nLewis, RE, Lewis, JH and Harrison, DL.1973. On a collection of mammals from Northern Saudi Arabia. Journal of Zoology, 144:61-74.\n\nLópez-Antoñanzas, R and Sen, S. 2006. New Saudi Arabian Miocene jumping mouse (Zapodidae): systematics and phylogeny. Journal of Vertebrate Paleontology, 26: 170-181.\n\nOrrock, JL, Danielson, BJ and Brinkerhoff, RJ. 2004. Rodent foraging is affected by indirect, but not by direct, cues of predation risk. Behavioral Ecology, 15: 433–437.\n\nReichman, O J. 1979. Desert granivore foraging and its impact on seed densities and distributions. Ecology, 60:1085-92.\n\nSarli, J, Lutermann, H, Alagaili, AN, Mohammed, OB and Bennett, NC. 2016. Seasonal reproduction in the Arabian spiny mouse, Acomys dimidiatus (Rodentia: Muridae) from Saudi Arabia: The role of rainfall and temperature. Journal of Arid Environments, 124:352-359.\n\nStrauss, MW, Shah, MS and Shobrak, M. 2008. Rodent trapping in the Saja/ Umm Ar-Rimth protected area of Saudi Arabia using two different trap types. Zoology in the Middle East, 43:31-39.",
    "output": "[['references', 'verbatimScientificName', 'taxonRank', 'verbatimTraitName', 'verbatimTraitUnit', 'individualCount', 'measurementValue_min', 'measurementValue_max', 'dispersion', 'statisticalMethod', 'verbatimTraitValue', 'sex', 'lifeStage', 'measurementMethod', 'measurementRemarks', 'measurementAccuracy', 'measurementDeterminedBy', 'verbatimLocality', 'author', 'associatedReferences'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Acomys dimidiatus', 'Species', 'Wt. g: weight in grams', 'g', '19', '', '', '', '', '41.22', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbillus dasyurus', 'Species', 'Wt. g: weight in grams', 'g', '1', '', '', '', '', '18', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbilus cheesmani', 'Species', 'Wt. g: weight in grams', 'g', '3', '', '', '', '', '23.5', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbilus nanus', 'Species', 'Wt. g: weight in grams', 'g', '2', '', '', '', '', '14.5', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Jaculus jaculus', 'Species', 'Wt. g: weight in grams', 'g', '6', '', '', '', '', '48.5', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Meriones libycus', 'Species', 'Wt. g: weight in grams', 'g', '17', '', '', '', '', '146.1', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Meriones crassus', 'Species', 'Wt. g: weight in grams', 'g', '8', '', '', '', '', '80.33', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Mus musculus', 'Species', 'Wt. g: weight in grams', 'g', '2', '', '', '', '', '7.6', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Rattus rattus', 'Species', 'Wt. g: weight in grams', 'g', '4', '', '', '', '', '161', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Acomys dimidiatus', 'Species', 'HB: Head and body length', 'mm', '19', '', '', '', '', '109.49', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbillus dasyurus', 'Species', 'HB: Head and body length', 'mm', '1', '', '', '', '', '84.12', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbilus cheesmani', 'Species', 'HB: Head and body length', 'mm', '3', '', '', '', '', '89.22', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbilus nanus', 'Species', 'HB: Head and body length', 'mm', '2', '', '', '', '', '77.12', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Jaculus jaculus', 'Species', 'HB: Head and body length', 'mm', '6', '', '', '', '', '114.21', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Meriones libycus', 'Species', 'HB: Head and body length', 'mm', '17', '', '', '', '', '167.2', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Meriones crassus', 'Species', 'HB: Head and body length', 'mm', '8', '', '', '', '', '126.88', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Mus musculus', 'Species', 'HB: Head and body length', 'mm', '2', '', '', '', '', '65.66', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Rattus rattus', 'Species', 'HB: Head and body length', 'mm', '4', '', '', '', '', '184.14', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Acomys dimidiatus', 'Species', 'TL: Tail length', 'mm', '19', '', '', '', '', '96.52', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbillus dasyurus', 'Species', 'TL: Tail length', 'mm', '1', '', '', '', '', '114.43', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbilus cheesmani', 'Species', 'TL: Tail length', 'mm', '3', '', '', '', '', '116.45', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbilus nanus', 'Species', 'TL: Tail length', 'mm', '2', '', '', '', '', '103.75', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Jaculus jaculus', 'Species', 'TL: Tail length', 'mm', '6', '', '', '', '', '168.16', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Meriones libycus', 'Species', 'TL: Tail length', 'mm', '17', '', '', '', '', '156.5', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Meriones crassus', 'Species', 'TL: Tail length', 'mm', '8', '', '', '', '', '138.73', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Mus musculus', 'Species', 'TL: Tail length', 'mm', '2', '', '', '', '', '69.85', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Rattus rattus', 'Species', 'TL: Tail length', 'mm', '4', '', '', '', '', '185.84', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Acomys dimidiatus', 'Species', 'HL: Head length', 'mm', '19', '', '', '', '', '33.46', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbillus dasyurus', 'Species', 'HL: Head length', 'mm', '1', '', '', '', '', '27.77', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbilus cheesmani', 'Species', 'HL: Head length', 'mm', '3', '', '', '', '', '29.59', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbilus nanus', 'Species', 'HL: Head length', 'mm', '2', '', '', '', '', '27.5', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Jaculus jaculus', 'Species', 'HL: Head length', 'mm', '6', '', '', '', '', '34.07', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Meriones libycus', 'Species', 'HL: Head length', 'mm', '17', '', '', '', '', '44.1', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Meriones crassus', 'Species', 'HL: Head length', 'mm', '8', '', '', '', '', '38.13', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Mus musculus', 'Species', 'HL: Head length', 'mm', '2', '', '', '', '', '21.21', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Rattus rattus', 'Species', 'HL: Head length', 'mm', '4', '', '', '', '', '39.98', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Acomys dimidiatus', 'Species', 'HW: Head width', 'mm', '19', '', '', '', '', '15.51', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbillus dasyurus', 'Species', 'HW: Head width', 'mm', '1', '', '', '', '', '14.52', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbilus cheesmani', 'Species', 'HW: Head width', 'mm', '3', '', '', '', '', '16.04', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbilus nanus', 'Species', 'HW: Head width', 'mm', '2', '', '', '', '', '14.45', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Jaculus jaculus', 'Species', 'HW: Head width', 'mm', '6', '', '', '', '', '25.79', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Meriones libycus', 'Species', 'HW: Head width', 'mm', '17', '', '', '', '', '23.5', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Meriones crassus', 'Species', 'HW: Head width', 'mm', '8', '', '', '', '', '21.29', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Mus musculus', 'Species', 'HW: Head width', 'mm', '2', '', '', '', '', '10.42', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Rattus rattus', 'Species', 'HW: Head width', 'mm', '4', '', '', '', '', '21.71', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Acomys dimidiatus', 'Species', 'E: Ear length', 'mm', '19', '', '', '', '', '17.25', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbillus dasyurus', 'Species', 'E: Ear length', 'mm', '1', '', '', '', '', '12.5', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbilus cheesmani', 'Species', 'E: Ear length', 'mm', '3', '', '', '', '', '12.01', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbilus nanus', 'Species', 'E: Ear length', 'mm', '2', '', '', '', '', '9.01', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Jaculus jaculus', 'Species', 'E: Ear length', 'mm', '6', '', '', '', '', '19.66', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Meriones libycus', 'Species', 'E: Ear length', 'mm', '17', '', '', '', '', '17.3', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Meriones crassus', 'Species', 'E: Ear length', 'mm', '8', '', '', '', '', '14.01', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Mus musculus', 'Species', 'E: Ear length', 'mm', '2', '', '', '', '', '12.72', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Rattus rattus', 'Species', 'E: Ear length', 'mm', '4', '', '', '', '', '24.93', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Acomys dimidiatus', 'Species', 'HF: Hindfoot length', 'mm', '19', '', '', '', '', '20.45', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbillus dasyurus', 'Species', 'HF: Hindfoot length', 'mm', '1', '', '', '', '', '20.52', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbilus cheesmani', 'Species', 'HF: Hindfoot length', 'mm', '3', '', '', '', '', '25.675', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbilus nanus', 'Species', 'HF: Hindfoot length', 'mm', '2', '', '', '', '', '23.54', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Jaculus jaculus', 'Species', 'HF: Hindfoot length', 'mm', '6', '', '', '', '', '60.72', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Meriones libycus', 'Species', 'HF: Hindfoot length', 'mm', '17', '', '', '', '', '41.38', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Meriones crassus', 'Species', 'HF: Hindfoot length', 'mm', '8', '', '', '', '', '33.78', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Mus musculus', 'Species', 'HF: Hindfoot length', 'mm', '2', '', '', '', '', '18.51', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Rattus rattus', 'Species', 'HF: Hindfoot length', 'mm', '4', '', '', '', '', '35.45', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Acomys dimidiatus', 'Species', 'IC: Interorbital constriction', 'mm', '19', '', '', '', '', '4.93', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbillus dasyurus', 'Species', 'IC: Interorbital constriction', 'mm', '1', '', '', '', '', '5.23', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbilus cheesmani', 'Species', 'IC: Interorbital constriction', 'mm', '3', '', '', '', '', '5.2', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbilus nanus', 'Species', 'IC: Interorbital constriction', 'mm', '2', '', '', '', '', '5.04', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Jaculus jaculus', 'Species', 'IC: Interorbital constriction', 'mm', '6', '', '', '', '', '11.01', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Meriones crassus', 'Species', 'IC: Interorbital constriction', 'mm', '8', '', '', '', '', '5.86', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Meriones libycus', 'Species', 'IC: Interorbital constriction', 'mm', '17', '', '', '', '', '6.85', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Mus musculus', 'Species', 'IC: Interorbital constriction', 'mm', '2', '', '', '', '', '3.59', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Rattus rattus', 'Species', 'IC: Interorbital constriction', 'mm', '4', '', '', '', '', '5.95', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Acomys dimidiatus', 'Species', 'ZB: Zygomatic breadth', 'mm', '19', '', '', '', '', '13.71', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbillus dasyurus', 'Species', 'ZB: Zygomatic breadth', 'mm', '1', '', '', '', '', '11.03', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbilus cheesmani', 'Species', 'ZB: Zygomatic breadth', 'mm', '3', '', '', '', '', '12.67', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbilus nanus', 'Species', 'ZB: Zygomatic breadth', 'mm', '2', '', '', '', '', '11.57', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Jaculus jaculus', 'Species', 'ZB: Zygomatic breadth', 'mm', '6', '', '', '', '', '20.61', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Meriones crassus', 'Species', 'ZB: Zygomatic breadth', 'mm', '8', '', '', '', '', '15.34', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Meriones libycus', 'Species', 'ZB: Zygomatic breadth', 'mm', '17', '', '', '', '', '17.4', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Mus musculus', 'Species', 'ZB: Zygomatic breadth', 'mm', '2', '', '', '', '', '9.01', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Rattus rattus', 'Species', 'ZB: Zygomatic breadth', 'mm', '4', '', '', '', '', '18.37', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Acomys dimidiatus', 'Species', 'BB: Brain case breadth', 'mm', '19', '', '', '', '', '11.2', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbillus dasyurus', 'Species', 'BB: Brain case breadth', 'mm', '1', '', '', '', '', '12.73', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbilus cheesmani', 'Species', 'BB: Brain case breadth', 'mm', '3', '', '', '', '', '9.67', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbilus nanus', 'Species', 'BB: Brain case breadth', 'mm', '2', '', '', '', '', '12.83', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Jaculus jaculus', 'Species', 'BB: Brain case breadth', 'mm', '6', '', '', '', '', '22.5', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Meriones crassus', 'Species', 'BB: Brain case breadth', 'mm', '8', '', '', '', '', '13.23', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Meriones libycus', 'Species', 'BB: Brain case breadth', 'mm', '17', '', '', '', '', '14.61', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Mus musculus', 'Species', 'BB: Brain case breadth', 'mm', '2', '', '', '', '', '7.94', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Rattus rattus', 'Species', 'BB: Brain case breadth', 'mm', '4', '', '', '', '', '13.52', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Acomys dimidiatus', 'Species', 'GtL: Greatest length of skull', 'mm', '19', '', '', '', '', '30.41', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbillus dasyurus', 'Species', 'GtL: Greatest length of skull', 'mm', '1', '', '', '', '', '26.42', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbilus cheesmani', 'Species', 'GtL: Greatest length of skull', 'mm', '3', '', '', '', '', '21.42', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbilus nanus', 'Species', 'GtL: Greatest length of skull', 'mm', '2', '', '', '', '', '28.9', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Jaculus jaculus', 'Species', 'GtL: Greatest length of skull', 'mm', '6', '', '', '', '', '31.9', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Meriones crassus', 'Species', 'GtL: Greatest length of skull', 'mm', '8', '', '', '', '', '30.86', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Meriones libycus', 'Species', 'GtL: Greatest length of skull', 'mm', '17', '', '', '', '', '36.82', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Rattus rattus', 'Species', 'GtL: Greatest length of skull', 'mm', '4', '', '', '', '', '41.11', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Acomys dimidiatus', 'Species', 'CBL', 'mm', '19', '', '', '', '', '28.83', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbillus dasyurus', 'Species', 'CBL', 'mm', '1', '', '', '', '', '23.32', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbilus cheesmani', 'Species', 'CBL', 'mm', '3', '', '', '', '', '19.24', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbilus nanus', 'Species', 'CBL', 'mm', '2', '', '', '', '', '25.8', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Jaculus jaculus', 'Species', 'CBL', 'mm', '6', '', '', '', '', '28.1', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Meriones crassus', 'Species', 'CBL', 'mm', '8', '', '', '', '', '28.53', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Meriones libycus', 'Species', 'CBL', 'mm', '17', '', '', '', '', '34.43', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Mus musculus', 'Species', 'CBL', 'mm', '2', '', '', '', '', '19.97', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Rattus rattus', 'Species', 'CBL', 'mm', '4', '', '', '', '', '38.92', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Acomys dimidiatus', 'Species', 'MXC: Maxillary cheekteeth', 'mm', '19', '', '', '', '', '4.79', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbillus dasyurus', 'Species', 'MXC: Maxillary cheekteeth', 'mm', '1', '', '', '', '', '3.79', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbilus cheesmani', 'Species', 'MXC: Maxillary cheekteeth', 'mm', '3', '', '', '', '', '3.72', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbilus nanus', 'Species', 'MXC: Maxillary cheekteeth', 'mm', '2', '', '', '', '', '3.98', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Jaculus jaculus', 'Species', 'MXC: Maxillary cheekteeth', 'mm', '6', '', '', '', '', '4.65', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Meriones crassus', 'Species', 'MXC: Maxillary cheekteeth', 'mm', '8', '', '', '', '', '5.24', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Meriones libycus', 'Species', 'MXC: Maxillary cheekteeth', 'mm', '17', '', '', '', '', '5.54', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Mus musculus', 'Species', 'MXC: Maxillary cheekteeth', 'mm', '2', '', '', '', '', '3.16', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Rattus rattus', 'Species', 'MXC: Maxillary cheekteeth', 'mm', '4', '', '', '', '', '6.55', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Acomys dimidiatus', 'Species', 'MDC: Mandibular cheekteeth', 'mm', '19', '', '', '', '', '4.47', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbillus dasyurus', 'Species', 'MDC: Mandibular cheekteeth', 'mm', '1', '', '', '', '', '3.13', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbilus cheesmani', 'Species', 'MDC: Mandibular cheekteeth', 'mm', '3', '', '', '', '', '3.72', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbilus nanus', 'Species', 'MDC: Mandibular cheekteeth', 'mm', '2', '', '', '', '', '3.75', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Jaculus jaculus', 'Species', 'MDC: Mandibular cheekteeth', 'mm', '6', '', '', '', '', '4.46', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Meriones crassus', 'Species', 'MDC: Mandibular cheekteeth', 'mm', '8', '', '', '', '', '5.12', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Meriones libycus', 'Species', 'MDC: Mandibular cheekteeth', 'mm', '17', '', '', '', '', '6.57', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Mus musculus', 'Species', 'MDC: Mandibular cheekteeth', 'mm', '2', '', '', '', '', '2.77', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Rattus rattus', 'Species', 'MDC: Mandibular cheekteeth', 'mm', '4', '', '', '', '', '6.16', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Acomys dimidiatus', 'Species', 'M: Mandible length (incisor included)', 'mm', '19', '', '', '', '', '18.25', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbillus dasyurus', 'Species', 'M: Mandible length (incisor included)', 'mm', '1', '', '', '', '', '14.22', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbilus cheesmani', 'Species', 'M: Mandible length (incisor included)', 'mm', '3', '', '', '', '', '15.34', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Gerbilus nanus', 'Species', 'M: Mandible length (incisor included)', 'mm', '2', '', '', '', '', '12.48', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Jaculus jaculus', 'Species', 'M: Mandible length (incisor included)', 'mm', '6', '', '', '', '', '18.7', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Meriones crassus', 'Species', 'M: Mandible length (incisor included)', 'mm', '8', '', '', '', '', '19.06', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Meriones libycus', 'Species', 'M: Mandible length (incisor included)', 'mm', '17', '', '', '', '', '20.36', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Mus musculus', 'Species', 'M: Mandible length (incisor included)', 'mm', '2', '', '', '', '', '11.28', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R., Al-Zein, M.S. and Lebanon, B., 2022. Rodents Diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal of Natural History, 9(1), pp.83-88.', 'Rattus rattus', 'Species', 'M: Mandible length (incisor included)', 'mm', '4', '', '', '', '', '24.46', '', '', '', '', '', '', 'Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia', '0000-0001-9627-8821', 'Original study']]"
  },
  {
    "instruction": "\nYou're an expert text parser. Your goal is to parse TSV fields from the input text. A single input text file can have multiple \ndifferent species and/or multiple individuals from the same species. The output should include a TSV header and the following\nfields should be filled for each species with information found from the input text:\n\nreferences:\nDescription: Complete bibliographic reference formatted according to Harvard citation style.\nExamples: Smith, J., Doe, A. and Jones, B. (2020) 'Plant functional traits across biomes', Journal of Ecology, 108(4), pp. 1200-1215.\n\nverbatimScientificName:\nDescription: The full, original scientific name as supplied in the raw data, including authorship or subspecific rank if recorded.\nExamples: Pinus sylvestris L., Panthera leo (Linnaeus, 1758)\n\ntaxonRank:\nDescription: Taxonomic rank of the target organism.\nExamples: species, genus, subspecies\n\nverbatimTraitName:\nDescription: Exact trait name as originally recorded in the source data or publication.\nExamples: SLA, specific leaf area, body mass\n\nverbatimTraitUnit:\nDescription: Original unit of measurement as stated in the source.\nExamples: mm, g, mm2/mg, kg\n\nindividualCount:\nDescription: Number of individuals aggregated into the reported measurement.\nExamples: 1, 15, 100\n\nmeasurementValue_min:\nDescription: Minimum measured value within a range or aggregate group.\nExamples: 12.4, 0.5\n\nmeasurementValue_max:\nDescription: Maximum measured value within a range or aggregate group.\nExamples: 28.1, 3.2\n\ndispersion:\nDescription: Measure of variation/dispersion associated with the central value (e.g., standard deviation, standard error).\nExamples: SD = 1.2, 0.45\n\nstatisticalMethod:\nDescription: Statistical metric used to summarize the values.\nExamples: mean, median, mode, range\n\nverbatimTraitValue:\nDescription: Raw value or text string as originally reported (numerical or categorical).\nExamples: 15.4, herbaceous, red\n\nsex:\nDescription: Biological sex of the specimen(s).\nExamples: male, female, hermaphrodite, mixed\n\nlifeStage:\nDescription: Developmental age or life stage of the organism.\nExamples: adult, juvenile, seedling, larva\n\nmeasurementMethod:\nDescription: Protocols, instruments, or procedures used to capture the measurement.\nExamples: Swinbank dry weight method, Caliper measurement of tarsus length\n\nmeasurementRemarks:\nDescription: Free-text notes or qualifying details about the measurement process or state of the specimen.\nExamples: Measured on dried museum specimens, Leaves collected from upper canopy\n\nmeasurementAccuracy:\nDescription: Precision, error tolerance, or accuracy rating of the instrument/method used.\nExamples: ± 0.01 g, 0.1 mm\n\nmeasurementDeterminedBy:\nDescription: Name or ORCID of the person/entity who made or validated the measurement.\nExamples: Jane Doe, https://orcid.org/0000-0000-0000-0000\n\nverbatimLocality:\nDescription: Original description of the sampling location before standardized georeferencing.\nExamples: Mount Rainier National Park, trail to Paradise, Plot 4B, Black Forest\n\nauthor:\nDescription: This field has to only include bioparser\nExamples: BioParser, BioParser, BioParser\n\nassociatedReferences:\nDescription: Citations, DOIs, or bibliographic references supporting the dataset or measurement.\nExamples: https://doi.org/10.1111/1365-2435.12345, Smith et al. 2020\n\nFULL OUTPUT EXAMPLE:\n[['references', 'verbatimScientificName', 'taxonRank', 'verbatimTraitName', 'verbatimTraitUnit', 'individualCount', 'measurementValue_min', 'measurementValue_max', 'dispersion', 'statisticalMethod', 'verbatimTraitValue', 'sex', 'lifeStage', 'measurementMethod', 'measurementRemarks', 'measurementAccuracy', 'measurementDeterminedBy', 'verbatimLocality', 'author', 'associatedReferences'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'BM = Body Mass in grams (g)', 'g ', '', '4.9', '5.8', '', '', '5.2', 'male', '', '', '', '', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'BM = Body Mass in grams (g)', 'g ', '', '4.5', '5.4', '', '', '4.8', 'female', '', '', '', '', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'TBL = Total Body Length', 'mm ', '', '152', '175', '', '', '166.3', 'male', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'WS = Wing Span (width of the bat)', 'mm ', '', '152', '175', '', '', '166.3', 'male', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'WS = Wing Span (width of the bat)', 'mm ', '', '118', '168', '', '', '149.5', 'female', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'FA = Forearm length', 'mm', '', '32', '34.6', '', '', '33.2', 'male', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'FA = Forearm length', 'mm', '', '32.4', '33.3', '', '', '32.9', 'female', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'D5 = 5th Digit length', 'mm', '', '40', '44.5', '', '', '41.9', 'male', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'D5 = 5th Digit length', 'mm ', '', '40', '41.4', '', '', '40.5', 'female', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'D3 = 3rd Digit length', 'mm ', '', '54', '58.6', '', '', '56.2', 'male', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'D3 = 3rd Digit length', 'mm ', '', '54.1', '57.2', '', '', '55.1', 'female', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'TL = Tail Length', 'mm ', '', '27.5', '36.4', '', '', '30.7', 'male', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'TL = Tail Length', 'mm ', '', '21.4', '31.5', '', '', '27.8', 'female', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'TBL = Total Body Length', 'mm ', '63', '60.3', '76.4', '', '', '', '', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'TL = Tail Length', 'mm ', '63', '21.4', '36.4', '', '', '', '', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'WS = Wing Span (width of the bat)', 'mm ', '63', '118', '175', '', '', '', '', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'FA = Forearm length', 'mm ', '63', '32', '34.6', '', '', '', '', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'D5 = 5th Digit length', 'mm ', '63', '40', '44.5', '', '', '', '', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'D3 = 3rd Digit length', 'mm ', '63', '54', '61', '', '', '', '', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'BM = Body Mass in grams (g)', 'g ', '63', '4.5', '5.8', '', '', '', '', '', '', '', '', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study'], ['Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.', 'Pipistrellus kuhlii', 'Species', 'EL = Ear Length', 'mm ', '63', '8.2', '9.6', '', '', '', '', '', 'caliper', 'Benda et al. 2008', ' 0.1 mm', '', 'Governorates of Beni Suef and Minya, Egypt', '0000-0001-9627-8821', 'Original study']]\"\n",
    "input": "**NOTES ON GEOGRAPHIC DISTRIBUTION**\n\nCheck List 19 (6): 1061–1064 https://doi.org/10.15560/19.6.1061\n\nPENSOFT\n\n# The first documented record of Acomys dimidiatus (Cretzschmar, 1826), Eastern Spiny Mouse (Rodentia, Muridae) from Lebanon\n\nMounir R. Abi-Said<sup>1</sup>, 2\\*, Elie El Haddad<sup>3</sup>\n\n1 Department of Earth and Life Sciences, Faculty of Sciences II, Lebanese University, Fanar, Lebanon • [mabisaid9@gmail.com](mailto:mabisaid9@gmail.com) [https://orcid.org/0009-0007-0219-4768](https://orcid.org/0009-0007-0219-4768)\n\n2 Animal Encounter, Aley, Lebanon\n\n3 Nature Conservation Department, Society for the Protection of Nature in Lebanon, Beirut, Lebanon • elie.s.haddad@outlook.com \\* Corresponding author\n\n**Abstract.** We report the first documented record of Acomys dimidiatus (Cretzschmar, 1826) from Lebanon. An opportunistic trapping was carried on in Hima Ibl Al Saqi, southern Lebanon for five consecutive nights during the spring and fall of 2022. Four spiny mice were trapped, and their morphometric measurements were recorded. These measurements resemble those documented for this species in other parts of the Middle East. Our new record extends the known distribution of Acomys dimidiatus approximately 45 km away from its most northern known geographic range.\n\n**Keywords.** Geographic distribution, Sherman traps, Middle East\n\nAcademic editor: Abhijeet Bayani Received 4 July 2023, accepted 11 December 2023, published 20 December 2023\n\nAbi-Said MR, El Haddad E (2023) The first documented record of Acomys dimidiatus (Cretzschmar, 1826), Eastern Spiny Mouse (Rodentia, Muridae) from Lebanon. Check List 19 (6): 1061–1064. https://doi.org/10.15560/19.6.1061\n\n## Introduction\n\nEastern Spiny Mouse, Acomys dimidiatus (Cretzschmar, 1826), is a small, nocturnal species with a wide distribution ranging from the western Sahara to East Africa, Arabia to Iran, and Pakistan (Harrison and Bates 1991; Nowak and Walker 1999; Denys 2017). Acomys dimidiatus is well distributed in Palestine (Qumsiyeh 1996), Jordan (Amr et al. 2018), and the Kingdom of Saudi Arabia (Buttiker and Harrison 1982; Abi-Said and Al-Zein 2022). It is also recorded in the United Arab Emirates (Cunningham 2004), Oman (Harrison 1980), and Yemen (Harrison 1972). Despite its widespread distribution in Arabia (Amr et al. 2010), there is a lack of information about the distribution and ecology of A. dimidiatus in many parts of its range, including Lebanon. Lewis et al. (1967) reported the presence of this species from Ouadi Jilo in Lebanon (Fig. 2) based on a personal communication with Dr. Makram N. Kaiser, stating that he caught four individuals in Ouadi Jilo near Tyre, but without further details. Hence, Lewis et al. (1967) recommended further studies to document and clarify the status of this species.\n\n## Methods\n\nWe performed an opportunistic rodent trapping by setting five trapping stations of 15 XLK Folding Live Capture Kangaroo Rat Rodent Sherman traps (7.62 × 9.53 × 30.48 cm) spaced 2–4 m apart and distributed throughout the Hima.\n\nIbl Es-Saqi Hima, which is roughly 40 km from Ouadi Jilo where A. dimidiatus was first reported in Lebanon, is a complex and diverse eumediterranean site located on the western slopes of Mount Hermon (Fig. 2). The Hima comprises several distinct habitats. The 38-ha pine woodland is a dominant feature of the site, in addition to stony hillsides and scrubland habitats which are characterized by rocky outcroppings and sparse vegetation, providing critical habitat for a wide range of plant and animal species.\n\nThe traps were set over two seasons, in spring (May) and fall (September) 2022, for five consecutive nights. We set the traps in the afternoon of the first day and checked them at dawn the following day. For bait, we used canary feed mix, sunflower seed, bread, and peanut butter. Rodents caught were euthanized following the ethical guidelines for the use of wild animal speci men in research by the American Society of Mammalogists (Sikes et al. 2016), identified, sexed, measured using digital calipers, and weighed using a Pesola Spring Scale (precision ± 0.3%). In addition, muscle tissue was collected and preserved in 96% ethanol for later DNA analysis. We recorded external measurements, including body, tail, head width and length, and ear and foot length. Individuals caught were skinned; nonetheless, the skin was easily broken, and only one specimen was successfully preserved. It was deposited in the Lebanese University Natural History Museum (NHM-LU), Faculty of Sciences II – AlFanar. We cleaned the skulls of the skinned animals and recorded the measurements. The cranial and dental measurements included the greatest length of the skull (GTL), condylobasal length (CBL), zygomatic breadth (ZB), breadth of the braincase (BB), interorbital constriction (IC), postorbital constriction (PC), maxillary cheekteeth (MXC), mandibular cheekteeth (MDC), mandible length (M), tympanic bulla (BU), and rostrum length (RL).\n\nWe also recorded morphometric measurements (Table 1) and photographed all trapped individuals (Fig. 1).\n\n## Results\n\n## Acomys dimidiatus (Cretzschmar, 1826)\n\nEastern Spiny Mouse\n\nFigure 1\n\n**New records** (Fig. 2). LEBANON – **Nabatieh Governorate** •  Ibl Es-Saqi Hima; 33°21′39″N, 035°37′58″E; 700 m alt; 30.V.2022; M. Abi-Said & E. El Haddad leg.; live trapped; 1 adult ♀, NHM-LU M129-22 • ibid.; 20.IX.2022; 2 adult ♂ • ibid.; 21.IX.2022; 1 adult ♂.\n\nFour A. dimidiatus were trapped, one female in the spring and three males in the fall at just one trapping station, which is characterized by a rocky habitat of a Mediterranean forest. The female was larger and older than the three males. This was reflected in the wear of its teeth and its dull pelage color.\n\n![](images/page_1_image_10.jpg)\n\nFigure 1. Acomys dimidiatus trapped at Ibl Es-Saqi Hima.\n\nTable 1. External measurements of the four caught Acomys dimidiatus and the cranial and dental measurements.\n\n|  | Average | Min. | Max. |\n| --- | --- | --- | --- |\n| Weight (gm) | 37.03 | 27.5 | 47.5 |\n| Total length (mm) | 217.04 | 194.4 | 233.3 |\n| Tail length (mm) | 101.16 | 87.84 | 110.37 |\n| Hind-foot length (mm) | 21.70 | 21.45 | 22.05 |\n| Front-arm length (mm) | 21.75 | 19.12 | 26.69 |\n| Ear length (mm) | 17.78 | 16.00 | 19.75 |\n| Head width (mm) | 15.68 | 15.06 | 16.45 |\n| Head length (mm) | 33.95 | 32.09 | 35.45 |\n| Greatest skull length (mm) | 31.97 | 31.17 | 33.49 |\n| Condylobasal length (mm) | 30.35 | 29.95 | 31.10 |\n| Zygomatic breadth (mm) | 14.55 | 13.80 | 15.76 |\n| Breadth of the braincase (mm) | 11.86 | 11.40 | 12.28 |\n| Interorbital constriction breadth (mm) | 5.61 | 4.97 | 6.75 |\n| Postorbital constriction breadth (mm) | 13.62 | 13.14 | 13.67 |\n| Maxillary cheekteeth length (mm) | 4.80 | 4.76 | 4.81 |\n| Mandibular cheekteeth length (mm) | 4.55 | 4.42 | 4.64 |\n| Mandible length (mm) | 16.32 | 15.65 | 16.81 |\n| Tympanic bulla (mm) | 6.60 | 6.15 | 7.06 |\n| Rostrum length (mm) | 15.51 | 15.21 | 15.8 |\n| Crown area of first upper molar (mm) | 2.23 | 2.15 | 2.32 |\n\n**Identification.** Acomys dimidiatus is a robust mouse. The pelage color on the back is dark to light brown, and the abdomen is white with a very sharp line of demarcation. White patches are present on the posterior side of the ear base and under the eyes (Fig. 1). Rigid spines cover 75% of the back, from the mid-dorsal region to the base of the tail; at the center of the back, these spines measure 16.56 mm long at the center and 0.51 mm wide on average; the rest of the body is covered with soft fur. The tail, with large and easily visible scales, is shorter (101.16 mm) than the combined head and body length. The ears are large (18 mm long) and without hair. Both palms and soles are naked and yellow-brown. There are three pairs of mammae. The skull has a broad braincase (11.86 mm). The dental formula is 1/1 0/0 0/0 3/3, and the crown area of the first upper molar is very large (on average 2.23 mm). The external measurements, the cranial, and the skull measurements of the trapped individuals are summarized in Table 1.\n\n## Discussion\n\nEven though Acomys dimidiatus is a typical desert species (Pinheiro et al. 2018), it inhabits a continental and humid climate in Lebanon. In our Lebanese specimens, most morphometric measurements were similar or close to those reported elsewhere in the Middle East (Atallah 1976; Bates and Harison 1991; Mendelsshon and Yom-Tov 1996; Ghadirian et al. 2011; Abi-Said and Al-Zein 2022).\n\nOur new data represents the most northern range of distribution for this species. Its presence in Syria was never been confirmed (Bates and Harison 1991; Shehab et al. 2018). Although there have been several studies on small mammals recovered from owl pellets from western Beqaa (Abi-Said et al. 2021) and Chaddra-Akkar in northern Lebanon (Abi-Said et al. 2014), A. dimidiatus was not recorded in these owls’ diet. Acomys dimidiatus is a rare species in Lebanon, with restricted geographic distribution. Regarding its conservation status, it is still premature to assess this species of the IUCN national Red List for Lebanon pending further collections. Nonetheless, recommendations will be forwarded to the Ministry of Environment to consider it threatened and protect its habitat until further studies.\n\n![](images/page_2_image_2.jpg)\n\nFigure 2. Distribution of A. dimidiatus in Lebanon according to Lewis et al. (1967) (yellow circle) and the newly documented site reported here (yellow star).\n\nIt is imperative that additional studies be conducted in the mountain ranges extending northward from southern Lebanon, especially in the arid rocky regions in eastern Lebanon, to comprehensively comprehend the geographic range of A. dimidiatus. To discern its taxonomic status and compare the local population with others in the Middle East, genetic and molecular studies are highly recommended.\n\n## Acknowledgments\n\nWe thank Professor Boris Krystufek, Professor Zuhair Amr, Sarah Karam, and Karim Abi-Said for reviewing this manuscript, and Dr. Chadi Abdallah for providing the map. Mr. Hassan AlAshkar, Ms. Karen Gebray el, Tania Maroun, and Stephanie Ferando are thanked for their help during the field survey. This study was supported by the Society for the Protection of Nature in Lebanon Beirut, Lebanon; this financial support is gratefully acknowledged. Finally, we express gratitude to the editor and the reviewers for their constructive feedback and their time and effort invested in revising our manuscript.\n\n## Author Contributions\n\nConceptualization: MAS. Data curation: MAS, EEH. Formal analysis: MAS. Funding acquisition: MAS. Investigation: MAS, EEH. Methodology: MAS. Resources: MAS. Supervision: MAS. Visualization: MAS. Project administration: MAS. Validation: MAS. Writing – original draft: MAS. Writing – review and editing: MAS, EEH.\n\n## References\n\nAbi-Said MR, Al-Zein MS (2022) Rodents diversity in Wadi As Sulai, Riyadh Province, Kingdom of Saudi Arabia. Jordan Journal Of Natural History 9 (1): 83–88.\n\nAbi-Said, MR, Al Masri MT, Amr ZS (2021) Summer diet of the Long-Eared Owl, Asio otus (Linnaeus, 1758), in Aammiq Wetland, Western Beqaa, Lebanon. Sandgrouse 43: 277–280.\n\nAbi-Said, MR, Shehab AH, Amr ZS (2014) Diet of the Barn Owl, Tyto alba, from Chaddra-Akkar, Northern Lebanon. Jordan Journal of Biological Sciences 7: 109–112.\n\nAmr, Z, Shenbrot, G, Molur, S (2010). Acomys dimidiatus (Mediterranean assessment). The IUCN Red List of Threatened Species 2010: e.T136471A4296167.\n\nAmr ZS, Abu Mohammad A, Qumsiyeh M, Eid E (2018) Systematics, distribution and ecological analysis of rodents in Jordan. Zootaxa 4397 (1): 26. [https://doi.org/10.11646/zootaxa.4397.1.1](https://doi.org/10.11646/zootaxa.4397.1.1)\n\nAtallah SI (1967). A new species of spiny mouse (Acomys) from Jordan. Journal of Mammalogy 48 (2): 258–261.\n\nButtiker W, Harrison DL (1982) Mammals of Saudi Arabia. On a collection of Rodentia from Saudi Arabia. Fauna Saudi Arabia 4: 488–502.\n\nCunningham PL (2004) Checklist and status of the terrestrial mammals from the United Arab Emirates. Zoology of the Middle East 33 (1): 7–20. [https://doi.org/10.1080/09397140.2004.10638059](https://doi.org/10.1080/09397140.2004.10638059)\n\nDenys C (2017) Arabian Spiny Mouse Acomys dimidiatus. In: Wilson DE, Lacher TE Mittermeier RA (Eds.). Handbook of the mammals of the world, Vol. 7, Rodents II. Lynx Edicions, Barcelona, Spain, 1008 pp.\n\nGhadirian T, Mohammadi S, Ashrafzadeh MR, Najafabadi MS, Shahi T, Khaleghi Hamidi AH (2011) Distribution and taxonomic status of the Eastern Spiny mouse (Acomys dimidiatus) (Rodentia: Muridae) in Iran. Acta Zoologica Lituanica, 21 (3): 244–249.\n\nPinheiro G, Prata DF, Araújo IM, Tiscornia G (2018) The African spiny mouse (Acomys spp.) as an emerging model for development and regeneration. Laboratory Animals 52 (6): 565–576. [https://doi.org/10.1177/0023677218769921](https://doi.org/10.1177/0023677218769921)\n\nHarrison DL (1972) The mammals of Arabia: Lagomorpha and Rodentia, volume III. Ernest Benn, London, UK, 385–670.\n\nHarrison DL (1980) The scientific results of Oman flora and fauna survey 1977 (Dhofar). The mammals obtained in Dhofar by the 1977 Oman flora and fauna survey. Journal Oman Studies Special Report 2: 387–397.\n\nHarrison DL, Bates PJJ (1991) The Mammals of Arabia. Harrison Zoological Museum, Sevevoaks, UK, 253–256.\n\nLewis, RE, Lewis, DL, Attallah, SI (1967) A review of Lebanese mammals. Lagomorpha and Rodentia. Journal of Zoology 153: 45–70\n\nMendelssohn H, Yom-Tov Y (1999). Fauna Palaestina, Mammalia of Israel. Keterpress Enterprises, Jerusalem, Israel, 476 pp.\n\nNowak RM, Walker EP (1999) Walker’s mammals of the world. Baltimore: Johns Hopkins University Press, Baltimore, USA, 1999 pp.\n\nQumsiyeh MB (1996) Mammals of the Holy Land. Texas Tech University Press. Lubbock, USA, 389 pp.\n\nShehab AH, Amr ZS, Abu Baker MA (2018) Rodents of southwestern Syria. Acta Societas Zoologicae Bohemicae 82: 177–194.\n\nSikes RS, Animal Care and Use Committee of the American Society of Mammalogists (2016) 2016 Guidelines of the American Society of Mammalogists for the use of wild mammals in research and education. Journal of Mammalogy 97 (3): 663–688. [https://doi.org/10.1093/jmammal/gyw078](https://doi.org/10.1093/jmammal/gyw078)",
    "output": "[['references', 'verbatimScientificName', 'taxonRank', 'verbatimTraitName', 'verbatimTraitUnit', 'individualCount', 'measurementValue_min', 'measurementValue_max', 'dispersion', 'statisticalMethod', 'verbatimTraitValue', 'sex', 'lifeStage', 'measurementMethod', 'measurementRemarks', 'measurementAccuracy', 'measurementDeterminedBy', 'verbatimLocality', 'author', 'associatedReferences'], ['Abi-Said, M.R. and El Haddad, E., 2023. The first documented record of Acomys dimidiatus (Cretzschmar, 1826), Eastern Spiny Mouse (Rodentia, Muridae) from Lebanon. Check List, 19(6), pp.1061-1064.', 'Acomys dimidiatus', 'Species', 'Weight ', 'g', '4', '27.5', '47.5', '', '', '37.03', '', '', '', '', '', '', 'Hima Ibl Al Saqi, southern Lebanon', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R. and El Haddad, E., 2023. The first documented record of Acomys dimidiatus (Cretzschmar, 1826), Eastern Spiny Mouse (Rodentia, Muridae) from Lebanon. Check List, 19(6), pp.1061-1064.', 'Acomys dimidiatus', 'Species', 'Total length ', 'mm', '4', '194.4', '233.3', '', '', '217.04', '', '', '', '', '', '', 'Hima Ibl Al Saqi, southern Lebanon', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R. and El Haddad, E., 2023. The first documented record of Acomys dimidiatus (Cretzschmar, 1826), Eastern Spiny Mouse (Rodentia, Muridae) from Lebanon. Check List, 19(6), pp.1061-1064.', 'Acomys dimidiatus', 'Species', 'Tail length ', 'mm', '4', '87.84', '110.37', '', '', '101.16', '', '', '', '', '', '', 'Hima Ibl Al Saqi, southern Lebanon', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R. and El Haddad, E., 2023. The first documented record of Acomys dimidiatus (Cretzschmar, 1826), Eastern Spiny Mouse (Rodentia, Muridae) from Lebanon. Check List, 19(6), pp.1061-1064.', 'Acomys dimidiatus', 'Species', 'Hind-foot length ', 'mm', '4', '21.45', '22.05', '', '', '21.70', '', '', '', '', '', '', 'Hima Ibl Al Saqi, southern Lebanon', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R. and El Haddad, E., 2023. The first documented record of Acomys dimidiatus (Cretzschmar, 1826), Eastern Spiny Mouse (Rodentia, Muridae) from Lebanon. Check List, 19(6), pp.1061-1064.', 'Acomys dimidiatus', 'Species', 'Front-arm length ', 'mm', '4', '19.12', '26.69', '', '', '21.75', '', '', '', '', '', '', 'Hima Ibl Al Saqi, southern Lebanon', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R. and El Haddad, E., 2023. The first documented record of Acomys dimidiatus (Cretzschmar, 1826), Eastern Spiny Mouse (Rodentia, Muridae) from Lebanon. Check List, 19(6), pp.1061-1064.', 'Acomys dimidiatus', 'Species', 'Ear length ', 'mm', '4', '16.00', '19.75', '', '', '17.78', '', '', '', '', '', '', 'Hima Ibl Al Saqi, southern Lebanon', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R. and El Haddad, E., 2023. The first documented record of Acomys dimidiatus (Cretzschmar, 1826), Eastern Spiny Mouse (Rodentia, Muridae) from Lebanon. Check List, 19(6), pp.1061-1064.', 'Acomys dimidiatus', 'Species', 'Head width ', 'mm', '4', '15.06', '16.45', '', '', '15.68', '', '', '', '', '', '', 'Hima Ibl Al Saqi, southern Lebanon', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R. and El Haddad, E., 2023. The first documented record of Acomys dimidiatus (Cretzschmar, 1826), Eastern Spiny Mouse (Rodentia, Muridae) from Lebanon. Check List, 19(6), pp.1061-1064.', 'Acomys dimidiatus', 'Species', 'Head length ', 'mm', '4', '32.09', '35.45', '', '', '33.95', '', '', '', '', '', '', 'Hima Ibl Al Saqi, southern Lebanon', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R. and El Haddad, E., 2023. The first documented record of Acomys dimidiatus (Cretzschmar, 1826), Eastern Spiny Mouse (Rodentia, Muridae) from Lebanon. Check List, 19(6), pp.1061-1064.', 'Acomys dimidiatus', 'Species', 'Greatest skull length ', 'mm', '4', '31.17', '33.49', '', '', '31.97', '', '', '', '', '', '', 'Hima Ibl Al Saqi, southern Lebanon', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R. and El Haddad, E., 2023. The first documented record of Acomys dimidiatus (Cretzschmar, 1826), Eastern Spiny Mouse (Rodentia, Muridae) from Lebanon. Check List, 19(6), pp.1061-1064.', 'Acomys dimidiatus', 'Species', 'Condylobasal length ', 'mm', '4', '29.95', '31.10', '', '', '30.35', '', '', '', '', '', '', 'Hima Ibl Al Saqi, southern Lebanon', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R. and El Haddad, E., 2023. The first documented record of Acomys dimidiatus (Cretzschmar, 1826), Eastern Spiny Mouse (Rodentia, Muridae) from Lebanon. Check List, 19(6), pp.1061-1064.', 'Acomys dimidiatus', 'Species', 'Zygomatic breadth ', 'mm', '4', '13.80', '15.76', '', '', '14.55', '', '', '', '', '', '', 'Hima Ibl Al Saqi, southern Lebanon', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R. and El Haddad, E., 2023. The first documented record of Acomys dimidiatus (Cretzschmar, 1826), Eastern Spiny Mouse (Rodentia, Muridae) from Lebanon. Check List, 19(6), pp.1061-1064.', 'Acomys dimidiatus', 'Species', 'Breadth of the braincase ', 'mm', '4', '11.40', '12.28', '', '', '11.86', '', '', '', '', '', '', 'Hima Ibl Al Saqi, southern Lebanon', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R. and El Haddad, E., 2023. The first documented record of Acomys dimidiatus (Cretzschmar, 1826), Eastern Spiny Mouse (Rodentia, Muridae) from Lebanon. Check List, 19(6), pp.1061-1064.', 'Acomys dimidiatus', 'Species', 'Interorbital constriction breadth ', 'mm', '4', '4.97', '6.75', '', '', '5.61', '', '', '', '', '', '', 'Hima Ibl Al Saqi, southern Lebanon', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R. and El Haddad, E., 2023. The first documented record of Acomys dimidiatus (Cretzschmar, 1826), Eastern Spiny Mouse (Rodentia, Muridae) from Lebanon. Check List, 19(6), pp.1061-1064.', 'Acomys dimidiatus', 'Species', 'Postorbital constriction breadth ', 'mm', '4', '13.14', '13.67', '', '', '13.62', '', '', '', '', '', '', 'Hima Ibl Al Saqi, southern Lebanon', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R. and El Haddad, E., 2023. The first documented record of Acomys dimidiatus (Cretzschmar, 1826), Eastern Spiny Mouse (Rodentia, Muridae) from Lebanon. Check List, 19(6), pp.1061-1064.', 'Acomys dimidiatus', 'Species', 'Maxillary cheekteeth length ', 'mm', '4', '4.76', '4.81', '', '', '4.80', '', '', '', '', '', '', 'Hima Ibl Al Saqi, southern Lebanon', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R. and El Haddad, E., 2023. The first documented record of Acomys dimidiatus (Cretzschmar, 1826), Eastern Spiny Mouse (Rodentia, Muridae) from Lebanon. Check List, 19(6), pp.1061-1064.', 'Acomys dimidiatus', 'Species', 'Mandibular cheekteeth length ', 'mm', '4', '4.42', '4.64', '', '', '4.55', '', '', '', '', '', '', 'Hima Ibl Al Saqi, southern Lebanon', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R. and El Haddad, E., 2023. The first documented record of Acomys dimidiatus (Cretzschmar, 1826), Eastern Spiny Mouse (Rodentia, Muridae) from Lebanon. Check List, 19(6), pp.1061-1064.', 'Acomys dimidiatus', 'Species', 'Mandible length ', 'mm', '4', '15.65', '16.81', '', '', '16.32', '', '', '', '', '', '', 'Hima Ibl Al Saqi, southern Lebanon', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R. and El Haddad, E., 2023. The first documented record of Acomys dimidiatus (Cretzschmar, 1826), Eastern Spiny Mouse (Rodentia, Muridae) from Lebanon. Check List, 19(6), pp.1061-1064.', 'Acomys dimidiatus', 'Species', 'Tympanic bulla ', 'mm', '4', '6.15', '7.06', '', '', '6.60', '', '', '', '', '', '', 'Hima Ibl Al Saqi, southern Lebanon', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R. and El Haddad, E., 2023. The first documented record of Acomys dimidiatus (Cretzschmar, 1826), Eastern Spiny Mouse (Rodentia, Muridae) from Lebanon. Check List, 19(6), pp.1061-1064.', 'Acomys dimidiatus', 'Species', 'Rostrum length ', 'mm', '4', '15.21', '15.8', '', '', '15.51', '', '', '', '', '', '', 'Hima Ibl Al Saqi, southern Lebanon', '0000-0001-9627-8821', 'Original study'], ['Abi-Said, M.R. and El Haddad, E., 2023. The first documented record of Acomys dimidiatus (Cretzschmar, 1826), Eastern Spiny Mouse (Rodentia, Muridae) from Lebanon. Check List, 19(6), pp.1061-1064.', 'Acomys dimidiatus', 'Species', 'Crown area of first upper molar ', 'mm', '4', '2.15', '2.32', '', '', '2.23', '', '', '', '', '', '', 'Hima Ibl Al Saqi, southern Lebanon', '0000-0001-9627-8821', 'Original study']]"
  }
]

<a name="Data"></a>
### Data Prep
We'll be using a sampled dataset of handwritten maths formulas. The goal is to convert these images into a computer readable form - ie in LaTeX form, so we can render it. This can be very useful for complex formulas.

You can access the dataset [here](https://huggingface.co/datasets/unsloth/LaTeX_OCR). The full dataset is [here](https://huggingface.co/datasets/linxy/LaTeX_OCR).

In [22]:
from datasets import Dataset

# If your data is a list of dictionaries:
dataset = Dataset.from_list(test_data)

Let's take an overview look at the dataset. We shall see what the 3rd image is, and what caption it had.

In [23]:
dataset

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 3
})

We can also render the LaTeX in the browser directly!

In [ ]:
instruction = "Write the LaTeX representation for this image."

def convert_to_conversation(sample):
    conversation = [
        { "role": "user",
          "content" : [
            {"type" : "text",  "text"  : instruction},
            {"type" : "image", "image" : sample["image"]} ]
        },
        { "role" : "assistant",
          "content" : [
            {"type" : "text",  "text"  : sample["text"]} ]
        },
    ]
    return { "messages" : conversation }
pass

Let's convert the dataset into the "correct" format for finetuning:

In [ ]:
converted_dataset = [convert_to_conversation(sample) for sample in dataset]

We look at how the conversations are structured for the first example:

In [ ]:
converted_dataset[0]

{'messages': [{'role': 'user',
   'content': [{'type': 'text',
     'text': 'Write the LaTeX representation for this image.'},
    {'type': 'image',
     'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=160x40>}]},
  {'role': 'assistant',
   'content': [{'type': 'text',
     'text': '{ \\frac { N } { M } } \\in { \\bf Z } , { \\frac { M } { P } } \\in { \\bf Z } , { \\frac { P } { Q } } \\in { \\bf Z }'}]}]}

Let's first see before we do any finetuning what the model outputs for the first example!

In [ ]:
FastVisionModel.for_inference(model) # Enable for inference!

image = dataset[2]["image"]
instruction = "Write the LaTeX representation for this image."

messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

 $$H ^ { \prime } = \beta N \int d \lambda \left\{ \frac { 1 } { 2 \beta ^ { 2 } N ^ { 2 } } \partial _ { \lambda } \zeta ^ { \dagger } \partial _ { \lambda } \zeta + V ( \lambda ) \zeta ^ { \dagger } \zeta \right\} .$$<|im_end|>
<|endoftext|>


<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support `DPOTrainer` and `GRPOTrainer` for reinforcement learning!

We use our new `UnslothVisionDataCollator` which will help in our vision finetuning setup.

In [25]:
from trl import SFTTrainer, SFTConfig

max_seq_length = 2048
dtype = None
load_in_4bit = True # Use 4-bit quantization to save memory

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # Essential: Tells the model when the response ends

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Format the text and append the EOS token
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text": texts }

# Convert python list to HF Dataset and apply formatting
dataset = Dataset.from_list(test_data)
dataset = dataset.map(formatting_prompts_func, batched = True)

# 4. Set up the SFTTrainer for text (No vision collator needed)
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can speed up training if you have lots of short texts
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 30,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/3 [00:00<?, ? examples/s]

In [26]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
5.336 GB of memory reserved.


In [27]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3 | Num Epochs = 30 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 6,389,760 of 859,375,680 (0.74% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,1.297750
2,1.297750
3,1.260648
4,1.190779
5,1.107775
6,1.022456
7,0.941731
8,0.873708
9,0.808788
10,0.741881


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-30/tokenizer_config.json.


In [28]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

454.3452 seconds used for training.
7.57 minutes used for training.
Peak reserved memory = 5.336 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 36.641 %.
Peak reserved memory for training % of max memory = 0.0 %.


In [32]:
# Save the fine-tuned model (LoRA adapter)
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

Unsloth: Restored added_tokens_decoder metadata in lora_model/tokenizer_config.json.


['lora_model/processor_config.json']

<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!

We use `min_p = 0.1` and `temperature = 1.5`. Read this [Tweet](https://x.com/menhguin/status/1826132708508213629) for more information on why.

In [34]:
import re
from transformers import TextStreamer
from unsloth import FastLanguageModel

# 1. Load your fine-tuned model and tokenizer
# (If you restarted your notebook kernel, point 'model_name' to your saved output folder)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="lora_model",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

# 2. Enable native fast inference mode
FastLanguageModel.for_inference(model)

# 3. Ensure pad token is properly configured for generation
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

==((====))==  Unsloth 2026.10.10: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

In [36]:
print("Type:", type(tokenizer))
print("Processor:", tokenizer)

print("Has tokenizer:", hasattr(tokenizer, "tokenizer"))
if hasattr(tokenizer, "tokenizer"):
    print("Inner tokenizer:", type(tokenizer.tokenizer))
    print("EOS:", repr(tokenizer.tokenizer.eos_token))
    print("PAD:", repr(tokenizer.tokenizer.pad_token))

Type: <class 'transformers.models.qwen3_vl.processing_qwen3_vl.Qwen3VLProcessor'>
Processor: Qwen3VLProcessor:
- image_processor: Qwen2VLImageProcessorFast {
  "data_format": "channels_first",
  "do_convert_rgb": true,
  "do_normalize": true,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": [
    0.5,
    0.5,
    0.5
  ],
  "image_processor_type": "Qwen2VLImageProcessorFast",
  "image_std": [
    0.5,
    0.5,
    0.5
  ],
  "merge_size": 2,
  "patch_size": 16,
  "resample": 3,
  "rescale_factor": 0.00392156862745098,
  "size": {
    "longest_edge": 16777216,
    "shortest_edge": 65536
  },
  "temporal_patch_size": 2
}

- tokenizer: TokenizersBackend(name_or_path='lora_model', vocab_size=248044, model_max_length=262144, padding_side='left', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>', 'audio_bos_token': '<|audio_start|>', 'audio_eos_token': '<|audio_end|>', 'audio_token': '<|audio_pad|>', 'image_token': '<|image_pad|>', 'v

In [37]:
test = tokenizer(
    text="Hello world",
    return_tensors="pt",
)

print(test.keys())
print(tokenizer.decode(test["input_ids"][0]))

KeysView({'input_ids': tensor([[9419, 1814]]), 'attention_mask': tensor([[1, 1]])})
Hello world


In [38]:
print("Model type:", type(model))
print("Processor type:", type(tokenizer))
print("Processor files:", getattr(tokenizer, "name_or_path", "not available"))
print("Processor attributes:", [
    x for x in ("tokenizer", "image_processor", "chat_template")
    if hasattr(tokenizer, x)
])

Model type: <class 'peft.peft_model.PeftModelForCausalLM'>
Processor type: <class 'transformers.models.qwen3_vl.processing_qwen3_vl.Qwen3VLProcessor'>
Processor files: not available
Processor attributes: ['tokenizer', 'image_processor', 'chat_template']


In [39]:
import torch

prompt = "Hello world"

inputs = tokenizer(
    text=prompt,
    return_tensors="pt",
).to(model.device)

print("Input IDs:", inputs["input_ids"])
print("Model device:", model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=20,
    do_sample=False,
)

print("Generation succeeded")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Input IDs: tensor([[9419, 1814]], device='cuda:0')
Model device: cuda:0
Generation succeeded
Hello world

Hello world

Hello world

Hello world

Hello world

Hello world

Hello world

Hello


In [40]:
inputs = tokenizer(
    text=formatted_prompt,
    return_tensors="pt",
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=128,
    do_sample=False,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:

You're an expert text parser. Your goal is to parse TSV fields from the input text. A single input text file can have multiple 
different species and/or multiple individuals from the same species. The output should include a TSV header and the following
fields should be filled for each species with information found from the input text:

references:
Description: Complete bibliographic reference formatted according to Harvard citation style.
Examples: Smith, J., Doe, A. and Jones, B. (2020) 'Plant functional traits across biomes', Journal of Ecology, 108(4), pp. 1200-1215.

verbatimScientificName:
Description: The full, original scientific name as supplied in the raw data, including authorship or subspecific rank if recorded.
Examples: Pinus sylvestris L., Panthera leo (Linnaeus, 1758)

taxonRank:
Description: Ta

In [42]:


# Helper function to sanitize invisible/corrupted characters (fixes padding/encoding errors)
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return text
    text = text.replace('\xa0', ' ')
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
    return text

# 4. Define your Alpaca Prompt Template (must match training exactly)
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
"""

# 5. Pull a sample from your test data and sanitize it
sample = test_data[0] # Change index if you want a different row
my_instruction = clean_text(sample["instruction"])
my_input = clean_text(sample["input"])

# Format the prompt (leaving the Response field open)
formatted_prompt = alpaca_prompt.format(my_instruction, my_input)

# 6. Tokenize the input using the properly loaded tokenizer
inputs = tokenizer(
    text=formatted_prompt,
    return_tensors="pt",
).to(model.device)

# 7. Set up the streamer and generate the response
#text_streamer = TextStreamer(tokenizer, skip_prompt=True)

#_ = model.generate(
#    **inputs,
#    streamer=text_streamer,
#    max_new_tokens=1024, # High limit for TSV data output
#    use_cache=True,
#    temperature=0.1,     # Low temperature for deterministic tabular parsing
#    min_p=0.1
#)

outputs = model.generate(
    **inputs,
    max_new_tokens=1024,
    use_cache=True,
    do_sample=False,
)

# Decode only the newly generated tokens
new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
response = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True,
)

print(response)


Here is the TSV header and the corresponding data filled for each species:

references:
Abdel-Hamid, F. and Alqahtani, A.R., 2022. Morphological characters and feeding habits of insectivorous Kuhl’s pipistrelle bat Pipistrellus kuhlii Cretzschmar, 1830, in different riverine habitats. Saudi Journal of Biological Sciences, 29(12), p.103490.

verbatimScientificName:
Pipistrellus kuhlii

taxonRank:
Species

verbatimTraitName:
BM = Body Mass in grams (g)

verbatimTraitUnit:
g 

individualCount:
count

measurementValue_min:
4.5

measurementValue_max:
5.8

dispersion:
range

statisticalMethod:
mean

verbatimTraitValue:
weight

sex:
male

lifeStage:
adult

measurementMethod:
Caliper measurement
measurementRemarks:
Whole body mass measured on dead specimens
measurementAccuracy:
± 0.01 g
measurementDeterminedBy:
Abdel-Hamid, F. and Alqahtani, A.R., 2022
verbatimLocality:
Governorates of Beni Suef and Minya, Egypt
author:
BioParser
associatedReferences:
Original study

Full Output Example:
[['re

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
#model.save_pretrained("qwen_lora")  # Local saving
#tokenizer.save_pretrained("qwen_lora")
# model.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN") # Online saving

['qwen_lora/processor_config.json']

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
#if False:
#    from unsloth import FastVisionModel
#    model, tokenizer = FastVisionModel.from_pretrained(
#        model_name = "qwen_lora", # YOUR MODEL YOU USED FOR TRAINING
#        load_in_4bit = True, # Set to False for 16bit LoRA
#    )
#    FastVisionModel.for_inference(model) # Enable for inference!
#
#image = dataset[0]["image"]
#instruction = "Write the LaTeX representation for this image."
#
#messages = [
#    {"role": "user", "content": [
#        {"type": "image"},
#        {"type": "text", "text": instruction}
#    ]}
#]
#input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
#inputs = tokenizer(
#    image,
#    input_text,
#    add_special_tokens = False,
#    return_tensors = "pt",
#).to("cuda")
#
#from transformers import TextStreamer
#text_streamer = TextStreamer(tokenizer, skip_prompt = True)
#_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
#                   use_cache = True, temperature = 1.5, min_p = 0.1)

\frac { N } { M } \in { \bf Z } , \frac { M } { P } \in { \bf Z } , \frac { P } { Q } \in { \bf Z }<|im_end|>


### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Select ONLY 1 to save! (Both not needed!)

# Save locally to 16bit
if False: model.save_pretrained_merged("unsloth_finetune", tokenizer,)

# To export and save to your Hugging Face account
if False: model.push_to_hub_merged("YOUR_USERNAME/unsloth_finetune", tokenizer, token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("qwen_finetune", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/qwen_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN",
    )

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Looking to use Unsloth locally? Read our [Installation Guide](https://unsloth.ai/docs/get-started/install) for details on installing Unsloth on Windows, Docker, AMD, Intel GPUs.
2. Learn how to do Reinforcement Learning with our [RL Guide and notebooks](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide).
3. Read our guides and notebooks for [Text-to-speech (TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) and [vision](https://unsloth.ai/docs/basics/vision-fine-tuning) model support.
4. Explore our [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms) to find dedicated guides for each model.
5. Need help with Inference? Read our [Inference & Deployment page](https://unsloth.ai/docs/basics/inference-and-deployment) for details on using vLLM, llama.cpp, Ollama etc.

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️

  <b>This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)</b>
</div>